In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


Pandas: 2.3.3
NumPy: 2.0.1


In [2]:
DATASET_DIR = Path(r"C:\Users\MY PC\Desktop\dataset")

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

Train directory: C:\Users\MY PC\Desktop\dataset\train
Test directory: C:\Users\MY PC\Desktop\dataset\test


In [3]:
files = [
    TRAIN_DIR / "train_source1.tsv",
    TRAIN_DIR / "train_source2.tsv",
    TRAIN_DIR / "train_source3.tsv",
    TRAIN_DIR / "train_ground_truth.tsv",
]

for file in files:
    print(f"{file.name}: {file.exists()}")

train_source1.tsv: True
train_source2.tsv: True
train_source3.tsv: True
train_ground_truth.tsv: True


In [4]:
source1_path = TRAIN_DIR / "train_source1.tsv"

s1_sample = pd.read_csv(
    source1_path,
    sep="\t",
    nrows=5
)

print("Shape:", s1_sample.shape)
print("Columns:", s1_sample.columns.tolist())

display(s1_sample)


Shape: (5, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [5]:
SAMPLE_SIZE = 10_000

s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    nrows=SAMPLE_SIZE
)

s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    nrows=SAMPLE_SIZE
)

s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    nrows=SAMPLE_SIZE
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

S1: (10000, 4)
S2: (10000, 4)
S3: (10000, 4)


In [6]:
gt = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    nrows=10
)

print("Shape:", gt.shape)
print("Columns:", gt.columns.tolist())

display(gt)

Shape: (10, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [7]:
import re
import unicodedata

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower()
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value

In [8]:
test_names = s1["business_name"].head(10).tolist()

for name in test_names:
    print(f"Original : {name}")
    print(f"Normalized: {normalize_text(name)}")
    print("-" * 60)

Original : Orelee's Barbershop
Normalized: orelee s barbershop
------------------------------------------------------------
Original : Prime Money
Normalized: prime money
------------------------------------------------------------
Original : B+ Retail Inc
Normalized: b retail inc
------------------------------------------------------------
Original : Christ Chapel
Normalized: christ chapel
------------------------------------------------------------
Original : Prabhav Business Center
Normalized: prabhav business center
------------------------------------------------------------
Original : Custom Wealth Services LLC
Normalized: custom wealth services llc
------------------------------------------------------------
Original : Consulting Nyasa Nursing Private Limited
Normalized: consulting nyasa nursing private limited
------------------------------------------------------------
Original : Nexus Anchor Rain
Normalized: nexus anchor rain
--------------------------------------------------

In [10]:
for df in [s1, s2, s3]:
    df["name_normalized"] = df["business_name"].apply(normalize_text)
    df["address_normalized"] = df["business_address"].apply(normalize_text)
    df["country_normalized"] = df["country"].apply(normalize_text)


display(
    s1[
        [
            "entity_id",
            "business_name",
            "name_normalized",
            "business_address",
            "address_normalized",
            "country"
        ]
    ].head(10)
)

,entity_id,business_name,name_normalized,business_address,address_normalized,country
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US
1,S1-773889195,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US
2,S1-377745466,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US
3,S1-133037285,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US
4,S1-755362802,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US


In [11]:
def name_prefix_key(row, n=4):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return None

    return f"{country}|{name[:n]}"


s1["block_name_prefix"] = s1.apply(name_prefix_key, axis=1)
s2["block_name_prefix"] = s2.apply(name_prefix_key, axis=1)
s3["block_name_prefix"] = s3.apply(name_prefix_key, axis=1)

display(
    s1[
        [
            "entity_id",
            "business_name",
            "country",
            "block_name_prefix"
        ]
    ].head(10)
)

,entity_id,business_name,country,block_name_prefix
0,S1-925783039,Orelee's Barbershop,US,us|orel
1,S1-773889195,Prime Money,US,us|prim
2,S1-377745466,B+ Retail Inc,US,us|b re
3,S1-133037285,Christ Chapel,US,us|chri
4,S1-755362802,Prabhav Business Center,India,india|prab
5,S1-851869949,Custom Wealth Services LLC,US,us|cust
6,S1-785847572,Consulting Nyasa Nursing Private Limited,India,india|cons
7,S1-27541239,Nexus Anchor Rain,US,us|nexu
8,S1-629417405,Moore Bitwise Inc,US,us|moor
9,S1-22305073,Dermatology Green Medicine,US,us|derm


In [12]:
from collections import defaultdict

def build_name_prefix_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        key = row["block_name_prefix"]
        if key:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        key = row["block_name_prefix"]
        if key:
            index[key].append(row["entity_id"])

    return index


name_prefix_index = build_name_prefix_index(s2, s3)

print("Number of blocking keys:", len(name_prefix_index))

Number of blocking keys: 6600


In [13]:
def generate_name_prefix_candidates(s1_row, index):
    key = s1_row["block_name_prefix"]

    if not key:
        return []

    return list(set(index.get(key, [])))


first_s1 = s1.iloc[0]

candidates = generate_name_prefix_candidates(
    first_s1,
    name_prefix_index
)

print("S1 entity:", first_s1["entity_id"])
print("Business:", first_s1["business_name"])
print("Blocking key:", first_s1["block_name_prefix"])
print("Number of candidates:", len(candidates))
print("Candidates:", candidates[:20])

S1 entity: S1-925783039
Business: Orelee's Barbershop
Blocking key: us|orel
Number of candidates: 2
Candidates: ['S2-762023797', 'S2-470535992']


In [14]:
def generate_all_name_prefix_candidates(s1, index):
    results = []

    for _, row in s1.iterrows():
        candidates = generate_name_prefix_candidates(row, index)

        results.append({
            "source1_entity_id": row["entity_id"],
            "candidate_entity_ids": candidates
        })

    return pd.DataFrame(results)


candidate_results = generate_all_name_prefix_candidates(
    s1,
    name_prefix_index
)

candidate_results.head()

,source1_entity_id,candidate_entity_ids
0,S1-925783039,"[S2-762023797, S2-470535992]"
1,S1-773889195,"[S3-201206588, S3-443516445, S2-432723670, S3-..."
2,S1-377745466,[]
3,S1-133037285,"[S3-593510333, S2-247565679, S3-559887621, S3-..."
4,S1-755362802,"[S2-832324901, S3-600731496]"


In [15]:
candidate_results["candidate_count"] = (
    candidate_results["candidate_entity_ids"]
    .apply(len)
)

print("Average candidates per S1:",
      candidate_results["candidate_count"].mean())

print("Maximum candidates for any S1:",
      candidate_results["candidate_count"].max())

print("Minimum candidates for any S1:",
      candidate_results["candidate_count"].min())

print(
    candidate_results["candidate_count"].describe()
)

Average candidates per S1: 13.8694
Maximum candidates for any S1: 176
Minimum candidates for any S1: 0
count    10000.000000
mean        13.869400
std         19.179737
min          0.000000
25%          1.000000
50%          7.000000
75%         20.000000
max        176.000000
Name: candidate_count, dtype: float64


In [16]:
def address_prefix_key(row, n=6):
    address = row["address_normalized"]
    country = row["country_normalized"]

    if not address:
        return None

    return f"{country}|{address[:n]}"

for df in [s1, s2, s3]:
    df["block_address_prefix"] = df.apply(
        address_prefix_key,
        axis=1
    )

s1[
    [
        "entity_id",
        "business_address",
        "address_normalized",
        "block_address_prefix"
    ]
].head(10)

,entity_id,business_address,address_normalized,block_address_prefix
0,S1-925783039,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,us|1795 w
1,S1-773889195,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,us|17560
2,S1-377745466,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,us|1712 m
3,S1-133037285,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,us|2100 c
4,S1-755362802,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,india|797 la
5,S1-851869949,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,us|oh col
6,S1-785847572,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,india|2505 t
7,S1-27541239,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,us|1111 c
8,S1-629417405,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,us|337 oa
9,S1-22305073,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,us|294 me


In [17]:
def build_address_prefix_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        key = row["block_address_prefix"]
        if key:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        key = row["block_address_prefix"]
        if key:
            index[key].append(row["entity_id"])

    return index


address_prefix_index = build_address_prefix_index(s2, s3)

print("Number of address blocking keys:", len(address_prefix_index))

Number of address blocking keys: 15257


In [21]:
#17
def generate_address_prefix_candidates(s1_row, index):
    key = s1_row["block_address_prefix"]

    if not key:
        return []

    return list(set(index.get(key, [])))

first_s1 = s1.iloc[0]

address_candidates = generate_address_prefix_candidates(
    first_s1,
    address_prefix_index
)

print("S1 entity:", first_s1["entity_id"])
print("Business:", first_s1["business_name"])
print("Address:", first_s1["business_address"])
print("Blocking key:", first_s1["block_address_prefix"])
print("Number of address candidates:", len(address_candidates))
print("Candidates:", address_candidates[:20])

S1 entity: S1-925783039
Business: Orelee's Barbershop
Address: 1795 Westchester Drive, High Point, NC
Blocking key: us|1795 w
Number of address candidates: 0
Candidates: []


In [22]:
#18
address_candidate_results = []

for _, row in s1.iterrows():
    candidates = generate_address_prefix_candidates(
        row,
        address_prefix_index
    )

    address_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

address_candidate_results = pd.DataFrame(
    address_candidate_results
)

address_candidate_results["candidate_count"] = (
    address_candidate_results["candidate_entity_ids"]
    .apply(len)
)

print(
    "Average candidates per S1:",
    address_candidate_results["candidate_count"].mean()
)

print(
    "Maximum candidates:",
    address_candidate_results["candidate_count"].max()
)

print(
    "Minimum candidates:",
    address_candidate_results["candidate_count"].min()
)

print(
    address_candidate_results["candidate_count"].describe()
)

Average candidates per S1: 9.981
Maximum candidates: 375
Minimum candidates: 0
count    10000.000000
mean         9.981000
std         38.670779
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        375.000000
Name: candidate_count, dtype: float64


In [27]:
#19
def generate_combined_candidates(
    s1_row,
    name_index,
    address_index
):
    name_candidates = generate_name_prefix_candidates(
        s1_row,
        name_index
    )

    address_candidates = generate_address_prefix_candidates(
        s1_row,
        address_index
    )

    return list(set(name_candidates) | set(address_candidates))

In [28]:
combined_results = []

for _, row in s1.iterrows():
    candidates = generate_combined_candidates(
        row,
        name_prefix_index,
        address_prefix_index
    )

    combined_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

combined_results = pd.DataFrame(combined_results)

combined_results["candidate_count"] = (
    combined_results["candidate_entity_ids"]
    .apply(len)
)

combined_results.head()

,source1_entity_id,candidate_entity_ids,candidate_count
0,S1-925783039,"[S2-762023797, S2-470535992]",2
1,S1-773889195,"[S3-201206588, S3-443516445, S2-432723670, S3-...",61
2,S1-377745466,[S2-131264573],1
3,S1-133037285,"[S3-593510333, S2-247565679, S2-36732240, S2-8...",23
4,S1-755362802,"[S2-832324901, S3-600731496]",2


In [29]:
print(
    "Average candidates per S1:",
    combined_results["candidate_count"].mean()
)

print(
    "Maximum candidates:",
    combined_results["candidate_count"].max()
)

print(
    "Minimum candidates:",
    combined_results["candidate_count"].min()
)

print(
    combined_results["candidate_count"].describe()
)

Average candidates per S1: 23.8334
Maximum candidates: 429
Minimum candidates: 0
count    10000.000000
mean        23.833400
std         42.208198
min          0.000000
25%          1.000000
50%         11.000000
75%         27.000000
max        429.000000
Name: candidate_count, dtype: float64


In [30]:
#22
name_nonzero = (
    candidate_results["candidate_count"] > 0
).sum()

address_nonzero = (
    address_candidate_results["candidate_count"] > 0
).sum()

combined_nonzero = (
    combined_results["candidate_count"] > 0
).sum()

print("Name prefix - S1 with candidates:", name_nonzero)
print("Address prefix - S1 with candidates:", address_nonzero)
print("Combined - S1 with candidates:", combined_nonzero)

Name prefix - S1 with candidates: 7563
Address prefix - S1 with candidates: 2235
Combined - S1 with candidates: 8176


In [31]:
#23
def first_token_key(row):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return None

    tokens = name.split()

    if not tokens:
        return None

    return f"{country}|{tokens[0]}"

for df in [s1, s2, s3]:
    df["block_first_token"] = df.apply(
        first_token_key,
        axis=1
    )

s1[
    [
        "entity_id",
        "business_name",
        "name_normalized",
        "block_first_token"
    ]
].head(10)

,entity_id,business_name,name_normalized,block_first_token
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,us|orelee
1,S1-773889195,Prime Money,prime money,us|prime
2,S1-377745466,B+ Retail Inc,b retail inc,us|b
3,S1-133037285,Christ Chapel,christ chapel,us|christ
4,S1-755362802,Prabhav Business Center,prabhav business center,india|prabhav
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,us|custom
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,india|consulting
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,us|nexus
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,us|moore
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,us|dermatology


In [32]:
def build_first_token_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        key = row["block_first_token"]

        if key:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        key = row["block_first_token"]

        if key:
            index[key].append(row["entity_id"])

    return index


first_token_index = build_first_token_index(s2, s3)

print("Number of first-token blocking keys:", len(first_token_index))

Number of first-token blocking keys: 9067


In [33]:
#25
def generate_first_token_candidates(s1_row, index):
    key = s1_row["block_first_token"]

    if not key:
        return []

    return list(set(index.get(key, [])))

first_s1 = s1.iloc[0]

token_candidates = generate_first_token_candidates(
    first_s1,
    first_token_index
)

print("S1 entity:", first_s1["entity_id"])
print("Business:", first_s1["business_name"])
print("Blocking key:", first_s1["block_first_token"])
print("Number of candidates:", len(token_candidates))
print("Candidates:", token_candidates[:20])

S1 entity: S1-925783039
Business: Orelee's Barbershop
Blocking key: us|orelee
Number of candidates: 0
Candidates: []


In [34]:
first_token_results = []

for _, row in s1.iterrows():
    candidates = generate_first_token_candidates(
        row,
        first_token_index
    )

    first_token_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

first_token_results = pd.DataFrame(first_token_results)

first_token_results["candidate_count"] = (
    first_token_results["candidate_entity_ids"]
    .apply(len)
)

print(
    "Average candidates per S1:",
    first_token_results["candidate_count"].mean()
)

print(
    "Maximum candidates:",
    first_token_results["candidate_count"].max()
)

print(
    "Minimum candidates:",
    first_token_results["candidate_count"].min()
)

print(
    first_token_results["candidate_count"].describe()
)

print(
    "S1 with at least one candidate:",
    (
        first_token_results["candidate_count"] > 0
    ).sum()
)

Average candidates per S1: 10.1926
Maximum candidates: 176
Minimum candidates: 0
count    10000.000000
mean        10.192600
std         15.596398
min          0.000000
25%          0.000000
50%          3.000000
75%         16.000000
max        176.000000
Name: candidate_count, dtype: float64
S1 with at least one candidate: 6366


In [35]:
#27
def char_ngram_keys(row, n=3):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return []

    # Remove spaces so small spelling differences can still overlap
    name_compact = name.replace(" ", "")

    if len(name_compact) < n:
        return []

    ngrams = {
        name_compact[i:i+n]
        for i in range(len(name_compact) - n + 1)
    }

    return [
        f"{country}|{ngram}"
        for ngram in ngrams
    ]

for df in [s1, s2, s3]:
    df["block_char_ngrams"] = df.apply(
        char_ngram_keys,
        axis=1
    )

s1[
    [
        "entity_id",
        "business_name",
        "block_char_ngrams"
    ]
].head(5)

,entity_id,business_name,block_char_ngrams
0,S1-925783039,Orelee's Barbershop,"[us|rel, us|ber, us|rbe, us|bar, us|hop, us|rs..."
1,S1-773889195,Prime Money,"[us|ime, us|one, us|emo, us|ney, us|mon, us|me..."
2,S1-377745466,B+ Retail Inc,"[us|lin, us|ret, us|inc, us|bre, us|ail, us|et..."
3,S1-133037285,Christ Chapel,"[us|ris, us|stc, us|tch, us|cha, us|ist, us|ch..."
4,S1-755362802,Prabhav Business Center,"[india|cen, india|ess, india|nes, india|sce, i..."


In [38]:
#28
def build_char_ngram_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        for key in row["block_char_ngrams"]:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        for key in row["block_char_ngrams"]:
            index[key].append(row["entity_id"])

    return index


char_ngram_index = build_char_ngram_index(s2, s3)

print(
    "Number of character n-gram blocking keys:",
    len(char_ngram_index)
)

Number of character n-gram blocking keys: 18140


In [39]:
#29
def generate_char_ngram_candidates(s1_row, index):
    keys = s1_row["block_char_ngrams"]

    if not keys:
        return []

    candidates = set()

    for key in keys:
        candidates.update(index.get(key, []))

    return list(candidates)

first_s1 = s1.iloc[0]

ngram_candidates = generate_char_ngram_candidates(
    first_s1,
    char_ngram_index
)

print("S1 entity:", first_s1["entity_id"])
print("Business:", first_s1["business_name"])
print("Number of candidates:", len(ngram_candidates))
print("Candidates:", ngram_candidates[:20])

S1 entity: S1-925783039
Business: Orelee's Barbershop
Number of candidates: 2056
Candidates: ['S3-391554617', 'S3-553985674', 'S2-631807146', 'S2-706697011', 'S3-416418318', 'S3-104862763', 'S3-134585768', 'S3-990468229', 'S2-421518911', 'S2-793665137', 'S3-892296515', 'S2-251801751', 'S2-714077903', 'S3-640485933', 'S3-290631820', 'S3-617511857', 'S2-456670248', 'S2-613343223', 'S2-490814405', 'S3-876372479']


In [40]:
#30
char_ngram_results = []

for _, row in s1.iterrows():
    candidates = generate_char_ngram_candidates(
        row,
        char_ngram_index
    )

    char_ngram_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

char_ngram_results = pd.DataFrame(
    char_ngram_results
)

char_ngram_results["candidate_count"] = (
    char_ngram_results["candidate_entity_ids"]
    .apply(len)
)

print(
    "Average candidates per S1:",
    char_ngram_results["candidate_count"].mean()
)

print(
    "Maximum candidates:",
    char_ngram_results["candidate_count"].max()
)

print(
    "Minimum candidates:",
    char_ngram_results["candidate_count"].min()
)

print(
    char_ngram_results["candidate_count"].describe()
)

print(
    "S1 with at least one candidate:",
    (
        char_ngram_results["candidate_count"] > 0
    ).sum()
)

Average candidates per S1: 3007.4275
Maximum candidates: 7644
Minimum candidates: 3
count    10000.000000
mean      3007.427500
std       1358.078695
min          3.000000
25%       1940.750000
50%       3211.000000
75%       4097.250000
max       7644.000000
Name: candidate_count, dtype: float64
S1 with at least one candidate: 10000


In [41]:
#31
def char_ngram_keys_4(row, n=4):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return []

    name_compact = name.replace(" ", "")

    if len(name_compact) < n:
        return []

    ngrams = {
        name_compact[i:i+n]
        for i in range(len(name_compact) - n + 1)
    }

    return [
        f"{country}|{ngram}"
        for ngram in ngrams
    ]

for df in [s1, s2, s3]:
    df["block_char_ngrams_4"] = df.apply(
        char_ngram_keys_4,
        axis=1
    )

s1[
    [
        "entity_id",
        "business_name",
        "block_char_ngrams_4"
    ]
].head(5)

,entity_id,business_name,block_char_ngrams_4
0,S1-925783039,Orelee's Barbershop,"[us|shop, us|eesb, us|ersh, us|elee, us|rele, ..."
1,S1-773889195,Prime Money,"[us|oney, us|prim, us|imem, us|emon, us|mone, ..."
2,S1-377745466,B+ Retail Inc,"[us|ilin, us|bret, us|tail, us|reta, us|aili, ..."
3,S1-133037285,Christ Chapel,"[us|hape, us|tcha, us|stch, us|rist, us|apel, ..."
4,S1-755362802,Prabhav Business Center,"[india|ssce, india|avbu, india|essc, india|usi..."


In [42]:
#32
def build_char_ngram_4_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        for key in row["block_char_ngrams_4"]:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        for key in row["block_char_ngrams_4"]:
            index[key].append(row["entity_id"])

    return index


char_ngram_4_index = build_char_ngram_4_index(s2, s3)

print(
    "Number of 4-gram blocking keys:",
    len(char_ngram_4_index)
)

Number of 4-gram blocking keys: 62213


In [44]:
#33
def generate_char_ngram_4_candidates(s1_row, index):
    keys = s1_row["block_char_ngrams_4"]

    if not keys:
        return []

    candidates = set()

    for key in keys:
        candidates.update(index.get(key, []))

    return list(candidates)

first_s1 = s1.iloc[0]

ngram_4_candidates = generate_char_ngram_4_candidates(
    first_s1,
    char_ngram_4_index
)

print("S1 entity:", first_s1["entity_id"])
print("Business:", first_s1["business_name"])
print("Number of candidates:", len(ngram_4_candidates))
print("Candidates:", ngram_4_candidates[:20])

S1 entity: S1-925783039
Business: Orelee's Barbershop
Number of candidates: 121
Candidates: ['S3-468163464', 'S2-82987673', 'S3-599153605', 'S3-15617359', 'S3-34737707', 'S3-797505453', 'S3-104862763', 'S2-878352124', 'S3-164289844', 'S3-640485933', 'S2-990031995', 'S3-105959407', 'S3-894437360', 'S2-671736202', 'S2-987865398', 'S3-544543420', 'S2-159701054', 'S3-879688889', 'S2-470535992', 'S3-178421278']


In [45]:
#34
char_ngram_4_results = []

for _, row in s1.iterrows():
    candidates = generate_char_ngram_4_candidates(
        row,
        char_ngram_4_index
    )

    char_ngram_4_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

char_ngram_4_results = pd.DataFrame(
    char_ngram_4_results
)

char_ngram_4_results["candidate_count"] = (
    char_ngram_4_results["candidate_entity_ids"]
    .apply(len)
)

print(
    "Average candidates per S1:",
    char_ngram_4_results["candidate_count"].mean()
)

print(
    "Maximum candidates:",
    char_ngram_4_results["candidate_count"].max()
)

print(
    "Minimum candidates:",
    char_ngram_4_results["candidate_count"].min()
)

print(
    char_ngram_4_results["candidate_count"].describe()
)

print(
    "S1 with at least one candidate:",
    (
        char_ngram_4_results["candidate_count"] > 0
    ).sum()
)

Average candidates per S1: 1207.5653
Maximum candidates: 4139
Minimum candidates: 0
count    10000.000000
mean      1207.565300
std       1257.356621
min          0.000000
25%        286.000000
50%        645.000000
75%       1565.500000
max       4139.000000
Name: candidate_count, dtype: float64
S1 with at least one candidate: 9982


In [46]:
s1_ids = set(s1["entity_id"])

print("Number of S1 IDs:", len(s1_ids))

Number of S1 IDs: 10000


In [47]:
#36
gt_path = TRAIN_DIR / "train_ground_truth.tsv"

gt_matches = []

for chunk in pd.read_csv(
    gt_path,
    sep="\t",
    chunksize=100_000
):
    matched = chunk[
        chunk["source1_entity_id"].isin(s1_ids)
    ]

    if not matched.empty:
        gt_matches.append(matched)

gt_sample = pd.concat(
    gt_matches,
    ignore_index=True
)

print("Ground-truth rows found:", len(gt_sample))
print(gt_sample.head())

Ground-truth rows found: 10000
  source1_entity_id                                 matched_entity_ids
0      S1-424976769              S2-18126063,S3-443109567,S3-296472077
1      S1-874660306                          S2-279857860,S3-289365741
2      S1-539879230  S2-83796020,S3-945547250,S3-127246506,S3-11765...
3      S1-840168502                                                NaN
4      S1-716830359  S2-795817041,S2-306163710,S3-762649005,S3-7375...


In [48]:
#37
print(
    "Unique S1 IDs in ground truth:",
    gt_sample["source1_entity_id"].nunique()
)

print(
    "S1 IDs without a ground-truth row:",
    len(s1_ids - set(gt_sample["source1_entity_id"]))
)

gt_sample["matched_entity_ids"].head(20)

Unique S1 IDs in ground truth: 10000
S1 IDs without a ground-truth row: 0


0                 S2-18126063,S3-443109567,S3-296472077
1                             S2-279857860,S3-289365741
2     S2-83796020,S3-945547250,S3-127246506,S3-11765...
3                                                   NaN
4     S2-795817041,S2-306163710,S3-762649005,S3-7375...
5     S2-601177655,S2-809918482,S3-588852284,S3-3395...
6     S2-562117280,S2-845708664,S2-820221567,S3-6499...
7                              S2-362218588,S3-11966366
8                S2-199917158,S3-118565228,S3-523567654
9                                          S3-230664101
10                            S2-856926544,S3-673592452
11               S2-884418256,S2-807723118,S3-557707615
12    S2-649519831,S2-664447396,S2-593664813,S3-6553...
13    S2-695716096,S3-871697729,S3-477353991,S3-6394...
14                S2-19065105,S3-611565511,S3-290432487
15    S2-380593256,S3-338454829,S3-360060474,S3-2569...
16               S3-406340823,S3-410710723,S3-995009055
17                                              

In [49]:
#38
def parse_matched_ids(value):
    if pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    return {
        entity_id.strip()
        for entity_id in value.split(",")
        if entity_id.strip()
    }


gt_sample["true_match_ids"] = (
    gt_sample["matched_entity_ids"]
    .apply(parse_matched_ids)
)

gt_sample.head()

gt_sample["true_match_count"] = (
    gt_sample["true_match_ids"].apply(len)
)

print(
    gt_sample["true_match_count"].describe()
)

print(
    "S1 with at least one true match:",
    (gt_sample["true_match_count"] > 0).sum()
)

print(
    "S1 with no true match:",
    (gt_sample["true_match_count"] == 0).sum()
)

count    10000.000000
mean         3.475200
std          1.726581
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         11.000000
Name: true_match_count, dtype: float64
S1 with at least one true match: 9440
S1 with no true match: 560


In [50]:
#39
name_eval = candidate_results.merge(
    gt_sample[
        [
            "source1_entity_id",
            "true_match_ids"
        ]
    ],
    on="source1_entity_id",
    how="left"
)

name_eval.head()

,source1_entity_id,candidate_entity_ids,candidate_count,true_match_ids
0,S1-925783039,"[S2-762023797, S2-470535992]",2,"{S3-997698194, S2-517291332, S3-698172821, S2-..."
1,S1-773889195,"[S3-201206588, S3-443516445, S2-432723670, S3-...",61,"{S2-374107005, S3-202893869, S3-622232873, S3-..."
2,S1-377745466,[],0,"{S2-203037501, S3-402918963, S3-70942743, S3-8..."
3,S1-133037285,"[S3-593510333, S2-247565679, S3-559887621, S3-...",23,"{S3-940895145, S2-407207105, S3-33144658, S3-1..."
4,S1-755362802,"[S2-832324901, S3-600731496]",2,"{S3-440254853, S2-587230276}"


In [51]:
#40
def calculate_recall(row):
    true_ids = row["true_match_ids"]
    candidate_ids = set(row["candidate_entity_ids"])

    if not true_ids:
        return np.nan

    return len(true_ids & candidate_ids) / len(true_ids)


name_eval["candidate_recall"] = (
    name_eval.apply(calculate_recall, axis=1)
)

print(
    "Candidate recall:",
    name_eval["candidate_recall"].mean()
)

print(
    name_eval["candidate_recall"].value_counts().sort_index()
)

Candidate recall: 0.001636904761904762
candidate_recall
0.000000    9388
0.142857       2
0.166667       7
0.200000      10
0.250000      12
0.333333      12
0.500000       8
1.000000       1
Name: count, dtype: int64


In [52]:
#41
name_eval["all_matches_found"] = (
    name_eval["candidate_recall"] == 1.0
)
evaluated_name = name_eval[
    name_eval["true_match_ids"].apply(len) > 0
]

print(
    "S1 with all true matches captured:",
    evaluated_name["all_matches_found"].mean()
)
print(
    "S1 with zero true matches captured:",
    (
        evaluated_name["candidate_recall"] == 0
    ).mean()
)

S1 with all true matches captured: 0.0001059322033898305
S1 with zero true matches captured: 0.9944915254237288


In [64]:
#42
true_entity_ids = set()

for ids in gt_sample["true_match_ids"]:
    true_entity_ids.update(ids)

print("Total unique true S2/S3 IDs:", len(true_entity_ids))

Total unique true S2/S3 IDs: 34752


In [62]:
#43
true_s2_ids = {
    entity_id
    for entity_id in true_entity_ids
    if entity_id.startswith("S2-")
}

true_s3_ids = {
    entity_id
    for entity_id in true_entity_ids
    if entity_id.startswith("S3-")
}

print("Unique true S2 IDs:", len(true_s2_ids))
print("Unique true S3 IDs:", len(true_s3_ids))

Unique true S2 IDs: 16765
Unique true S3 IDs: 17987


In [63]:
#44
def extract_entities_by_id(file_path, target_ids, chunksize=100_000):
    chunks = []

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        chunksize=chunksize
    ):
        matched = chunk[
            chunk["entity_id"].isin(target_ids)
        ]

        if not matched.empty:
            chunks.append(matched)

    if not chunks:
        return pd.DataFrame()

    return pd.concat(chunks, ignore_index=True)

true_s2 = extract_entities_by_id(
    TRAIN_DIR / "train_source2.tsv",
    true_s2_ids
)

true_s3 = extract_entities_by_id(
    TRAIN_DIR / "train_source3.tsv",
    true_s3_ids
)

print("True S2 records extracted:", len(true_s2))
print("True S3 records extracted:", len(true_s3))

True S2 records extracted: 16765
True S3 records extracted: 17987


In [65]:
#45
for df in [true_s2, true_s3]:
    df["name_normalized"] = df["business_name"].apply(normalize_text)
    df["address_normalized"] = df["business_address"].apply(normalize_text)
    df["country_normalized"] = df["country"].apply(normalize_text)

print("True S2 columns:", true_s2.columns.tolist())
print("True S3 columns:", true_s3.columns.tolist())

True S2 columns: ['entity_id', 'business_name', 'business_address', 'country', 'name_normalized', 'address_normalized', 'country_normalized']
True S3 columns: ['entity_id', 'business_name', 'business_address', 'country', 'name_normalized', 'address_normalized', 'country_normalized']


In [66]:
#46
for df in [true_s2, true_s3]:
    df["block_name_prefix"] = df.apply(name_prefix_key, axis=1)
    df["block_address_prefix"] = df.apply(address_prefix_key, axis=1)
    df["block_first_token"] = df.apply(first_token_key, axis=1)
    df["block_char_ngrams"] = df.apply(char_ngram_keys, axis=1)
    df["block_char_ngrams_4"] = df.apply(char_ngram_keys_4, axis=1)

print("Blocking keys created.")

Blocking keys created.


In [67]:
#47
true_name_index = build_name_prefix_index(true_s2, true_s3)

print("True-entity name blocking keys:", len(true_name_index))

True-entity name blocking keys: 5545


In [68]:
#48
true_name_candidate_results = []

for _, row in s1.iterrows():
    candidates = generate_name_prefix_candidates(row, true_name_index)

    true_name_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

true_name_candidate_results = pd.DataFrame(true_name_candidate_results)

print(true_name_candidate_results.head())

  source1_entity_id                               candidate_entity_ids
0      S1-925783039  [S2-517291332, S2-157377754, S3-997698194, S3-...
1      S1-773889195  [S3-863654537, S2-682089546, S3-615053758, S3-...
2      S1-377745466         [S2-203037501, S3-402918963, S3-836170816]
3      S1-133037285  [S3-267248991, S3-33144658, S2-40421768, S3-50...
4      S1-755362802  [S3-888096642, S3-758288604, S2-572762497, S3-...


In [69]:
#49
true_name_eval = true_name_candidate_results.merge(
    gt_sample[["source1_entity_id", "true_match_ids"]],
    on="source1_entity_id",
    how="left"
)

true_name_eval["candidate_recall"] = true_name_eval.apply(
    calculate_recall,
    axis=1
)

print("Mean recall:", true_name_eval["candidate_recall"].mean())
print(
    "S1 with all true matches captured:",
    (true_name_eval["candidate_recall"] == 1.0).mean()
)
print(
    "S1 with at least one true match captured:",
    (true_name_eval["candidate_recall"] > 0).mean()
)

Mean recall: 0.7689396232298774
S1 with all true matches captured: 0.451
S1 with at least one true match captured: 0.9008


In [70]:
#50:
true_address_index = build_address_prefix_index(true_s2, true_s3)

print("True-entity address blocking keys:", len(true_address_index))

True-entity address blocking keys: 14415


In [71]:
#51
true_address_candidate_results = []

for _, row in s1.iterrows():
    candidates = generate_address_prefix_candidates(row, true_address_index)

    true_address_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

true_address_candidate_results = pd.DataFrame(
    true_address_candidate_results
)

print(true_address_candidate_results.head())

  source1_entity_id                               candidate_entity_ids
0      S1-925783039         [S3-997698194, S2-517291332, S2-157377754]
1      S1-773889195  [S3-161452820, S3-202893869, S3-189140669, S3-...
2      S1-377745466                       [S3-402918963, S3-836170816]
3      S1-133037285                       [S3-183080822, S2-407207105]
4      S1-755362802                                                 []


In [72]:
#52
true_address_eval = true_address_candidate_results.merge(
    gt_sample[["source1_entity_id", "true_match_ids"]],
    on="source1_entity_id",
    how="left"
)

true_address_eval["candidate_recall"] = true_address_eval.apply(
    calculate_recall,
    axis=1
)

print("Mean recall:", true_address_eval["candidate_recall"].mean())

print(
    "S1 with all true matches captured:",
    (true_address_eval["candidate_recall"] == 1.0).mean()
)

print(
    "S1 with at least one true match captured:",
    (true_address_eval["candidate_recall"] > 0).mean()
)

Mean recall: 0.4786824013439479
S1 with all true matches captured: 0.1768
S1 with at least one true match captured: 0.7039


In [73]:
#53
def generate_true_combined_candidates(s1_row, name_index, address_index):
    name_candidates = generate_name_prefix_candidates(
        s1_row, name_index
    )
    address_candidates = generate_address_prefix_candidates(
        s1_row, address_index
    )

    return list(set(name_candidates) | set(address_candidates))

In [74]:
#54
true_combined_candidate_results = []

for _, row in s1.iterrows():
    candidates = generate_true_combined_candidates(
        row,
        true_name_index,
        true_address_index
    )

    true_combined_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

true_combined_candidate_results = pd.DataFrame(
    true_combined_candidate_results
)

print(true_combined_candidate_results.head())

  source1_entity_id                               candidate_entity_ids
0      S1-925783039  [S2-517291332, S3-997698194, S2-157377754, S3-...
1      S1-773889195  [S2-15538892, S3-863654537, S2-706696711, S2-6...
2      S1-377745466         [S3-402918963, S3-836170816, S2-203037501]
3      S1-133037285  [S2-174648817, S3-940895145, S3-801589006, S2-...
4      S1-755362802  [S3-888096642, S3-758288604, S2-572762497, S3-...


In [75]:
#55
true_combined_eval = true_combined_candidate_results.merge(
    gt_sample[["source1_entity_id", "true_match_ids"]],
    on="source1_entity_id",
    how="left"
)

true_combined_eval["candidate_recall"] = true_combined_eval.apply(
    calculate_recall,
    axis=1
)

print("Mean recall:", true_combined_eval["candidate_recall"].mean())

print(
    "S1 with all true matches captured:",
    (true_combined_eval["candidate_recall"] == 1.0).mean()
)

print(
    "S1 with at least one true match captured:",
    (true_combined_eval["candidate_recall"] > 0).mean()
)

Mean recall: 0.880690577078289
S1 with all true matches captured: 0.653
S1 with at least one true match captured: 0.9283


In [76]:
#56
true_first_token_index = build_first_token_index(true_s2, true_s3)

print(
    "True-entity first-token blocking keys:",
    len(true_first_token_index)
)

True-entity first-token blocking keys: 8584


In [77]:
#57
true_first_token_candidate_results = []

for _, row in s1.iterrows():
    candidates = generate_first_token_candidates(
        row,
        true_first_token_index
    )

    true_first_token_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

true_first_token_candidate_results = pd.DataFrame(
    true_first_token_candidate_results
)

print(true_first_token_candidate_results.head())

  source1_entity_id                               candidate_entity_ids
0      S1-925783039  [S2-517291332, S2-157377754, S3-997698194, S3-...
1      S1-773889195  [S2-682089546, S2-76392328, S3-886833136, S2-1...
2      S1-377745466  [S2-703072292, S3-836170816, S3-355590735, S3-...
3      S1-133037285  [S3-940895145, S2-407207105, S3-252593505, S3-...
4      S1-755362802                                     [S2-587230276]


In [78]:
#58
true_first_token_eval = true_first_token_candidate_results.merge(
    gt_sample[["source1_entity_id", "true_match_ids"]],
    on="source1_entity_id",
    how="left"
)

true_first_token_eval["candidate_recall"] = (
    true_first_token_eval.apply(
        calculate_recall,
        axis=1
    )
)

print(
    "Mean recall:",
    true_first_token_eval["candidate_recall"].mean()
)

print(
    "S1 with all true matches captured:",
    (true_first_token_eval["candidate_recall"] == 1.0).mean()
)

print(
    "S1 with at least one true match captured:",
    (true_first_token_eval["candidate_recall"] > 0).mean()
)

Mean recall: 0.728740706092401
S1 with all true matches captured: 0.3762
S1 with at least one true match captured: 0.8926


In [79]:
#59
def generate_name_address_first_token_candidates(
    s1_row,
    name_index,
    address_index,
    first_token_index
):
    name_candidates = generate_name_prefix_candidates(
        s1_row, name_index
    )

    address_candidates = generate_address_prefix_candidates(
        s1_row, address_index
    )

    first_token_candidates = generate_first_token_candidates(
        s1_row, first_token_index
    )

    return list(
        set(name_candidates)
        | set(address_candidates)
        | set(first_token_candidates)
    )

In [80]:
#60
true_name_address_first_candidate_results = []

for _, row in s1.iterrows():
    candidates = generate_name_address_first_token_candidates(
        row,
        true_name_index,
        true_address_index,
        true_first_token_index
    )

    true_name_address_first_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

true_name_address_first_candidate_results = pd.DataFrame(
    true_name_address_first_candidate_results
)

print(true_name_address_first_candidate_results.head())

  source1_entity_id                               candidate_entity_ids
0      S1-925783039  [S2-517291332, S3-997698194, S2-157377754, S3-...
1      S1-773889195  [S2-15538892, S3-863654537, S2-706696711, S2-6...
2      S1-377745466  [S3-836170816, S2-703072292, S3-355590735, S3-...
3      S1-133037285  [S3-267248991, S3-33144658, S3-502311860, S2-4...
4      S1-755362802  [S3-888096642, S3-758288604, S2-572762497, S3-...


In [81]:
#61
true_name_address_first_eval = (
    true_name_address_first_candidate_results.merge(
        gt_sample[["source1_entity_id", "true_match_ids"]],
        on="source1_entity_id",
        how="left"
    )
)

true_name_address_first_eval["candidate_recall"] = (
    true_name_address_first_eval.apply(
        calculate_recall,
        axis=1
    )
)

print(
    "Mean recall:",
    true_name_address_first_eval["candidate_recall"].mean()
)

print(
    "S1 with all true matches captured:",
    (
        true_name_address_first_eval["candidate_recall"] == 1.0
    ).mean()
)

print(
    "S1 with at least one true match captured:",
    (
        true_name_address_first_eval["candidate_recall"] > 0
    ).mean()
)

Mean recall: 0.8833542675544794
S1 with all true matches captured: 0.6587
S1 with at least one true match captured: 0.9285


In [82]:
#62
combined_candidate_results = []

for _, row in s1.iterrows():
    candidates = generate_combined_candidates(
        row,
        name_prefix_index,
        address_prefix_index
    )

    combined_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": candidates
    })

combined_candidate_results = pd.DataFrame(
    combined_candidate_results
)

combined_counts = (
    combined_candidate_results["candidate_entity_ids"]
    .apply(len)
)

print("Name OR Address candidate statistics:")
print(combined_counts.describe())

print("Average candidates/S1:", combined_counts.mean())
print("Maximum candidates/S1:", combined_counts.max())
print("Minimum candidates/S1:", combined_counts.min())
print(
    "S1 with at least one candidate:",
    (combined_counts > 0).sum()
)

Name OR Address candidate statistics:
count    10000.000000
mean        23.833400
std         42.208198
min          0.000000
25%          1.000000
50%         11.000000
75%         27.000000
max        429.000000
Name: candidate_entity_ids, dtype: float64
Average candidates/S1: 23.8334
Maximum candidates/S1: 429
Minimum candidates/S1: 0
S1 with at least one candidate: 8176


In [83]:
#63
all_three_candidate_results = []

for _, row in s1.iterrows():
    candidates = (
        set(generate_name_prefix_candidates(row, name_prefix_index))
        | set(generate_address_prefix_candidates(row, address_prefix_index))
        | set(generate_first_token_candidates(row, first_token_index))
    )

    all_three_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": list(candidates)
    })

all_three_candidate_results = pd.DataFrame(
    all_three_candidate_results
)

all_three_counts = (
    all_three_candidate_results["candidate_entity_ids"]
    .apply(len)
)

print("Name OR Address OR First Token statistics:")
print(all_three_counts.describe())

print("Average candidates/S1:", all_three_counts.mean())
print("Maximum candidates/S1:", all_three_counts.max())
print("Minimum candidates/S1:", all_three_counts.min())
print(
    "S1 with at least one candidate:",
    (all_three_counts > 0).sum()
)

Name OR Address OR First Token statistics:
count    10000.000000
mean        24.102800
std         42.170547
min          0.000000
25%          2.000000
50%         11.000000
75%         28.000000
max        429.000000
Name: candidate_entity_ids, dtype: float64
Average candidates/S1: 24.1028
Maximum candidates/S1: 429
Minimum candidates/S1: 0
S1 with at least one candidate: 8361


In [84]:
#64
TOTAL_TARGET_RECORDS = len(s2) + len(s3)

name_address_reduction = (
    1 - combined_counts.mean() / TOTAL_TARGET_RECORDS
)

all_three_reduction = (
    1 - all_three_counts.mean() / TOTAL_TARGET_RECORDS
)

print("Total target records per S1:", TOTAL_TARGET_RECORDS)

print(
    "Name OR Address reduction ratio:",
    name_address_reduction
)

print(
    "Name OR Address OR First Token reduction ratio:",
    all_three_reduction
)

Total target records per S1: 20000
Name OR Address reduction ratio: 0.99880833
Name OR Address OR First Token reduction ratio: 0.99879486


In [85]:
#65
def extract_postal_code(value):
    if pd.isna(value):
        return ""

    text = str(value).upper()

    # Indian PIN / common 5-6 digit postal codes
    matches = re.findall(r"\b\d{5,6}\b", text)

    if not matches:
        return ""

    # Use the last numeric postal-looking token.
    return matches[-1]

In [86]:
#66
for df in [s1, true_s2, true_s3]:
    df["postal_code"] = df["business_address"].apply(
        extract_postal_code
    )

    df["block_postal"] = df.apply(
        lambda row: (
            f"{row['country_normalized']}|{row['postal_code']}"
            if row["postal_code"]
            else None
        ),
        axis=1
    )

print(
    s1[["business_address", "postal_code", "block_postal"]].head(10)
)

                                    business_address postal_code block_postal
0             1795 Westchester Drive, High Point, NC                     None
1                    17560 Ellis Road, Tahlequah, OK       17560     us|17560
2                1712 Montebello Avenue, Phoenix, AZ                     None
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD                     None
4  797, Lake Town Block A, Kolkata, Howrah, West ...                     None
5                  OH, Columbus, 5559 Orville Avenue                     None
6  2505, Tower 1, Oakwood, Runwal Greens, Mulund ...                     None
7       1111 Church Street, Unit 2007, Nashville, TN                     None
8              337 Oakland Avenue, Michigan City, IN                     None
9  294 Meadowcreek Drive, Unit Unit 2, Village Of...                     None


In [87]:
#67
def extract_postal_code(value):
    if pd.isna(value):
        return ""

    text = str(value).upper()

    # Look for a 6-digit Indian PIN.
    indian_pins = re.findall(r"\b[1-9]\d{5}\b", text)

    if indian_pins:
        return indian_pins[-1]

    # Look for a US ZIP code, but only when it appears
    # after a state abbreviation.
    us_zip = re.findall(
        r"\b[A-Z]{2}\s+(\d{5}(?:-\d{4})?)\b",
        text
    )

    if us_zip:
        return us_zip[-1]

    return ""

In [88]:
#68
for df in [s1, true_s2, true_s3]:
    df["postal_code"] = df["business_address"].apply(
        extract_postal_code
    )

    df["block_postal"] = df.apply(
        lambda row: (
            f"{row['country_normalized']}|{row['postal_code']}"
            if row["postal_code"]
            else None
        ),
        axis=1
    )

print(
    s1[["business_address", "postal_code", "block_postal"]].head(20)
)

                                     business_address postal_code block_postal
0              1795 Westchester Drive, High Point, NC                     None
1                     17560 Ellis Road, Tahlequah, OK                     None
2                 1712 Montebello Avenue, Phoenix, AZ                     None
3   2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD                     None
4   797, Lake Town Block A, Kolkata, Howrah, West ...                     None
5                   OH, Columbus, 5559 Orville Avenue                     None
6   2505, Tower 1, Oakwood, Runwal Greens, Mulund ...                     None
7        1111 Church Street, Unit 2007, Nashville, TN                     None
8               337 Oakland Avenue, Michigan City, IN                     None
9   294 Meadowcreek Drive, Unit Unit 2, Village Of...                     None
10            11643 Prosperity Road, South Jordan, UT                     None
11  H.No.16-11-23/37/A, 2Nd Floor, Flat No.207, Sa..

In [89]:
#69
def address_token_keys(row):
    address = row["address_normalized"]
    country = row["country_normalized"]

    if not address:
        return []

    tokens = address.split()

    # Remove very short/common tokens that create huge blocks.
    tokens = [
        token for token in tokens
        if len(token) >= 3
    ]

    # Use unique tokens.
    tokens = set(tokens)

    return [
        f"{country}|{token}"
        for token in tokens
    ]

In [90]:
#70
for df in [s1, true_s2, true_s3]:
    df["block_address_tokens"] = df.apply(
        address_token_keys,
        axis=1
    )

print(
    s1[
        [
            "business_address",
            "address_normalized",
            "block_address_tokens"
        ]
    ].head(10)
)

                                    business_address  \
0             1795 Westchester Drive, High Point, NC   
1                    17560 Ellis Road, Tahlequah, OK   
2                1712 Montebello Avenue, Phoenix, AZ   
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD   
4  797, Lake Town Block A, Kolkata, Howrah, West ...   
5                  OH, Columbus, 5559 Orville Avenue   
6  2505, Tower 1, Oakwood, Runwal Greens, Mulund ...   
7       1111 Church Street, Unit 2007, Nashville, TN   
8              337 Oakland Avenue, Michigan City, IN   
9  294 Meadowcreek Drive, Unit Unit 2, Village Of...   

                                  address_normalized  \
0               1795 westchester drive high point nc   
1                      17560 ellis road tahlequah ok   
2                  1712 montebello avenue phoenix az   
3     2100 cameron drive unit apartment g dundalk md   
4   797 lake town block a kolkata howrah west bengal   
5                    oh columbus 5559 orville a

In [91]:
#71
COMMON_ADDRESS_TOKENS = {
    "street", "st",
    "road", "rd",
    "avenue", "ave",
    "drive", "dr",
    "lane", "ln",
    "boulevard", "blvd",
    "highway", "hwy",
    "way",
    "place", "pl",
    "court", "ct",
    "circle", "cir",
    "parkway", "pkwy",
    "building", "block",
    "unit", "floor",
    "flat",
    "apartment",
    "apt",
    "house",
    "no",
    "number"
}

def address_token_keys_filtered(row):
    address = row["address_normalized"]
    country = row["country_normalized"]

    if not address:
        return []

    tokens = address.split()

    tokens = [
        token for token in tokens
        if len(token) >= 3
        and token not in COMMON_ADDRESS_TOKENS
        and not token.isdigit()
    ]

    return list({
        f"{country}|{token}"
        for token in tokens
    })

In [92]:
#72
for df in [s1, true_s2, true_s3]:
    df["block_address_tokens_filtered"] = df.apply(
        address_token_keys_filtered,
        axis=1
    )

print(
    s1[
        [
            "business_address",
            "block_address_tokens_filtered"
        ]
    ].head(10)
)

                                    business_address  \
0             1795 Westchester Drive, High Point, NC   
1                    17560 Ellis Road, Tahlequah, OK   
2                1712 Montebello Avenue, Phoenix, AZ   
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD   
4  797, Lake Town Block A, Kolkata, Howrah, West ...   
5                  OH, Columbus, 5559 Orville Avenue   
6  2505, Tower 1, Oakwood, Runwal Greens, Mulund ...   
7       1111 Church Street, Unit 2007, Nashville, TN   
8              337 Oakland Avenue, Michigan City, IN   
9  294 Meadowcreek Drive, Unit Unit 2, Village Of...   

                       block_address_tokens_filtered  
0                [us|high, us|westchester, us|point]  
1                           [us|ellis, us|tahlequah]  
2                        [us|montebello, us|phoenix]  
3                           [us|dundalk, us|cameron]  
4  [india|howrah, india|lake, india|west, india|t...  
5                          [us|orville, us|columbus] 

In [93]:
#73
true_address_token_index = defaultdict(list)

for _, row in true_s2.iterrows():
    for key in row["block_address_tokens_filtered"]:
        true_address_token_index[key].append(row["entity_id"])

for _, row in true_s3.iterrows():
    for key in row["block_address_tokens_filtered"]:
        true_address_token_index[key].append(row["entity_id"])

print(
    "True-entity address-token blocking keys:",
    len(true_address_token_index)
)

True-entity address-token blocking keys: 15529


In [94]:
#74
true_address_token_candidate_results = []

for _, row in s1.iterrows():
    candidates = set()

    for key in row["block_address_tokens_filtered"]:
        candidates.update(
            true_address_token_index.get(key, [])
        )

    true_address_token_candidate_results.append({
        "source1_entity_id": row["entity_id"],
        "candidate_entity_ids": list(candidates)
    })

true_address_token_candidate_results = pd.DataFrame(
    true_address_token_candidate_results
)

print(true_address_token_candidate_results.head())

  source1_entity_id                               candidate_entity_ids
0      S1-925783039  [S3-18836357, S2-271957796, S2-752007555, S2-7...
1      S1-773889195  [S2-374107005, S3-202893869, S3-622232873, S3-...
2      S1-377745466  [S2-228886754, S3-651892442, S2-200232988, S2-...
3      S1-133037285  [S3-940895145, S3-434710271, S3-491605587, S2-...
4      S1-755362802  [S3-338454829, S3-348621121, S2-911182333, S2-...


In [95]:
#75
true_address_token_eval = (
    true_address_token_candidate_results.merge(
        gt_sample[
            ["source1_entity_id", "true_match_ids"]
        ],
        on="source1_entity_id",
        how="left"
    )
)

true_address_token_eval["candidate_recall"] = (
    true_address_token_eval.apply(
        calculate_recall,
        axis=1
    )
)

print(
    "Mean recall:",
    true_address_token_eval["candidate_recall"].mean()
)

print(
    "S1 with all true matches captured:",
    (
        true_address_token_eval["candidate_recall"] == 1.0
    ).mean()
)

print(
    "S1 with at least one true match captured:",
    (
        true_address_token_eval["candidate_recall"] > 0
    ).mean()
)

Mean recall: 0.9499144975786926
S1 with all true matches captured: 0.7904
S1 with at least one true match captured: 0.9397


In [97]:
#76 Create filtered address-token keys for the 10k S2/S3 sample
for df in [s2, s3]:
    df["block_address_tokens_filtered"] = df.apply(
        address_token_keys_filtered,
        axis=1
    )

# Build index on the actual 10k + 10k candidate sample
address_token_index = defaultdict(list)

for _, row in s2.iterrows():
    for key in row["block_address_tokens_filtered"]:
        address_token_index[key].append(row["entity_id"])

for _, row in s3.iterrows():
    for key in row["block_address_tokens_filtered"]:
        address_token_index[key].append(row["entity_id"])

# Generate candidate counts
address_token_counts = []

for _, row in s1.iterrows():
    candidates = set()

    for key in row["block_address_tokens_filtered"]:
        candidates.update(
            address_token_index.get(key, [])
        )

    address_token_counts.append(len(candidates))

address_token_counts = pd.Series(address_token_counts)

print("Address-token candidate statistics:")
print(address_token_counts.describe())

print("Average candidates/S1:", address_token_counts.mean())
print("Maximum candidates/S1:", address_token_counts.max())
print("Minimum candidates/S1:", address_token_counts.min())
print(
    "S1 with at least one candidate:",
    (address_token_counts > 0).sum()
)

Address-token candidate statistics:
count    10000.000000
mean       505.691100
std        632.220004
min          0.000000
25%         24.000000
50%        165.000000
75%        855.000000
max       3367.000000
dtype: float64
Average candidates/S1: 505.6911
Maximum candidates/S1: 3367
Minimum candidates/S1: 0
S1 with at least one candidate: 9838


In [98]:
#77
def address_two_token_keys(row):
    address = row["address_normalized"]
    country = row["country_normalized"]

    if not address:
        return []

    tokens = [
        token
        for token in address.split()
        if len(token) >= 3
        and token not in COMMON_ADDRESS_TOKENS
        and not token.isdigit()
    ]

    tokens = list(dict.fromkeys(tokens))

    if len(tokens) < 2:
        return []

    keys = []

    for i in range(len(tokens)):
        for j in range(i + 1, len(tokens)):
            keys.append(
                f"{country}|{tokens[i]}|{tokens[j]}"
            )

    return keys

In [99]:
#78
for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_address_two_tokens"] = df.apply(
        address_two_token_keys,
        axis=1
    )

print(
    s1[
        [
            "business_address",
            "block_address_two_tokens"
        ]
    ].head(10)
)

                                    business_address  \
0             1795 Westchester Drive, High Point, NC   
1                    17560 Ellis Road, Tahlequah, OK   
2                1712 Montebello Avenue, Phoenix, AZ   
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD   
4  797, Lake Town Block A, Kolkata, Howrah, West ...   
5                  OH, Columbus, 5559 Orville Avenue   
6  2505, Tower 1, Oakwood, Runwal Greens, Mulund ...   
7       1111 Church Street, Unit 2007, Nashville, TN   
8              337 Oakland Avenue, Michigan City, IN   
9  294 Meadowcreek Drive, Unit Unit 2, Village Of...   

                            block_address_two_tokens  
0  [us|westchester|high, us|westchester|point, us...  
1                               [us|ellis|tahlequah]  
2                            [us|montebello|phoenix]  
3                               [us|cameron|dundalk]  
4  [india|lake|town, india|lake|kolkata, india|la...  
5                              [us|columbus|orville] 

In [103]:
#79

def generate_address_two_token_candidates(s1_row, index):
    keys = s1_row["block_address_two_tokens"]

    if not keys:
        return []

    candidates = set()

    for key in keys:
        candidates.update(index.get(key, []))

    return list(candidates)


address_two_token_recall_rows = []

for _, row in s1.iterrows():
    true_ids = gt_sample.loc[
        gt_sample["source1_entity_id"] == row["entity_id"],
        "true_match_ids"
    ]

    if len(true_ids) == 0:
        continue

    true_ids = true_ids.iloc[0]

    candidates = set(
        generate_address_two_token_candidates(
            row,
            address_two_token_true_index
        )
    )

    recall = (
        len(true_ids & candidates) / len(true_ids)
        if len(true_ids) > 0
        else 1.0
    )

    address_two_token_recall_rows.append({
        "entity_id": row["entity_id"],
        "true_match_count": len(true_ids),
        "captured_match_count": len(true_ids & candidates),
        "recall": recall
    })


address_two_token_recall_df = pd.DataFrame(
    address_two_token_recall_rows
)

print("Mean recall:",
      address_two_token_recall_df["recall"].mean())

print("All true matches captured:",
      (address_two_token_recall_df["recall"] == 1.0).mean())

print("At least one true match captured:",
      (address_two_token_recall_df["captured_match_count"] > 0).mean())

Mean recall: 0.7948951190476189
All true matches captured: 0.6014
At least one true match captured: 0.863


In [104]:
#80
address_two_token_sample_index = build_address_two_token_index(
    s2,
    s3
)

print("Number of blocking keys:", len(address_two_token_sample_index))

Number of blocking keys: 141138


In [105]:
#81
address_two_token_candidate_counts = []

for _, row in s1.iterrows():
    candidates = generate_address_two_token_candidates(
        row,
        address_two_token_sample_index
    )

    address_two_token_candidate_counts.append(
        len(candidates)
    )


address_two_token_candidate_counts = np.array(
    address_two_token_candidate_counts
)

print(
    "Average candidates per S1:",
    address_two_token_candidate_counts.mean()
)

print(
    "Maximum candidates per S1:",
    address_two_token_candidate_counts.max()
)

print(
    "Minimum candidates per S1:",
    address_two_token_candidate_counts.min()
)

print(
    "Std:",
    address_two_token_candidate_counts.std()
)

print(
    "25%:",
    np.percentile(address_two_token_candidate_counts, 25)
)

print(
    "Median:",
    np.percentile(address_two_token_candidate_counts, 50)
)

print(
    "75%:",
    np.percentile(address_two_token_candidate_counts, 75)
)

print(
    "S1 with at least one candidate:",
    (address_two_token_candidate_counts > 0).sum()
)

Average candidates per S1: 86.5395
Maximum candidates per S1: 948
Minimum candidates per S1: 0
Std: 152.67252418084271
25%: 0.0
Median: 2.0
75%: 114.0
S1 with at least one candidate: 5583


In [106]:
#82
total_possible_candidates = len(s2) + len(s3)

avg_candidates = address_two_token_candidate_counts.mean()

reduction_ratio = (
    1 - (avg_candidates / total_possible_candidates)
)

print("Total possible candidates per S1:", total_possible_candidates)
print("Average candidates per S1:", avg_candidates)
print("Reduction ratio:", reduction_ratio)
print("Reduction percentage:", reduction_ratio * 100)

Total possible candidates per S1: 20000
Average candidates per S1: 86.5395
Reduction ratio: 0.995673025
Reduction percentage: 99.5673025


In [107]:
#83
blocking_results = pd.DataFrame([
    {
        "strategy": "Name Prefix",
        "mean_recall": 0.7689396232,
        "all_matches_captured": 0.4510,
        "at_least_one_match": 0.9008,
        "avg_candidates": 13.8694,
        "max_candidates": 176,
        "reduction_ratio": 1 - (13.8694 / 20000)
    },
    {
        "strategy": "Address Prefix",
        "mean_recall": 0.4786824013,
        "all_matches_captured": 0.1768,
        "at_least_one_match": 0.7039,
        "avg_candidates": 9.9810,
        "max_candidates": 375,
        "reduction_ratio": 1 - (9.9810 / 20000)
    },
    {
        "strategy": "Name OR Address Prefix",
        "mean_recall": 0.8806905771,
        "all_matches_captured": 0.6530,
        "at_least_one_match": 0.9283,
        "avg_candidates": 23.8334,
        "max_candidates": 429,
        "reduction_ratio": 1 - (23.8334 / 20000)
    },
    {
        "strategy": "First Token",
        "mean_recall": 0.7287407061,
        "all_matches_captured": 0.3762,
        "at_least_one_match": 0.8926,
        "avg_candidates": 10.1926,
        "max_candidates": 176,
        "reduction_ratio": 1 - (10.1926 / 20000)
    },
    {
        "strategy": "Name OR Address OR First Token",
        "mean_recall": 0.8833542676,
        "all_matches_captured": 0.6587,
        "at_least_one_match": 0.9285,
        "avg_candidates": 24.1028,
        "max_candidates": 429,
        "reduction_ratio": 1 - (24.1028 / 20000)
    },
    {
        "strategy": "Address Token",
        "mean_recall": 0.9499144976,
        "all_matches_captured": 0.7904,
        "at_least_one_match": 0.9397,
        "avg_candidates": 505.6911,
        "max_candidates": 3367,
        "reduction_ratio": 1 - (505.6911 / 20000)
    },
    {
        "strategy": "Address Two-Token",
        "mean_recall": 0.7948951190,
        "all_matches_captured": 0.6014,
        "at_least_one_match": 0.8630,
        "avg_candidates": 86.5395,
        "max_candidates": 948,
        "reduction_ratio": 1 - (86.5395 / 20000)
    }
])

blocking_results

,strategy,mean_recall,all_matches_captured,at_least_one_match,avg_candidates,max_candidates,reduction_ratio
0,Name Prefix,0.768940,0.4510,0.9008,13.8694,176,0.999307
1,Address Prefix,0.478682,0.1768,0.7039,9.9810,375,0.999501
2,Name OR Address Prefix,0.880691,0.6530,0.9283,23.8334,429,0.998808
3,First Token,0.728741,0.3762,0.8926,10.1926,176,0.999490
4,Name OR Address OR First Token,0.883354,0.6587,0.9285,24.1028,429,0.998795
5,Address Token,0.949914,0.7904,0.9397,505.6911,3367,0.974715
6,Address Two-Token,0.794895,0.6014,0.8630,86.5395,948,0.995673


In [108]:
#84

def address_first_two_token_key(row):
    address = row["address_normalized"]
    country = row["country_normalized"]

    if not address:
        return None

    tokens = [
        token
        for token in address.split()
        if len(token) >= 3
        and token not in COMMON_ADDRESS_TOKENS
        and not token.isdigit()
    ]

    tokens = list(dict.fromkeys(tokens))

    if len(tokens) < 2:
        return None

    return f"{country}|{tokens[0]}|{tokens[1]}"


for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_address_first_two"] = df.apply(
        address_first_two_token_key,
        axis=1
    )

print(
    s1[
        [
            "business_address",
            "block_address_first_two"
        ]
    ].head(10)
)

                                    business_address block_address_first_two
0             1795 Westchester Drive, High Point, NC     us|westchester|high
1                    17560 Ellis Road, Tahlequah, OK      us|ellis|tahlequah
2                1712 Montebello Avenue, Phoenix, AZ   us|montebello|phoenix
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      us|cameron|dundalk
4  797, Lake Town Block A, Kolkata, Howrah, West ...         india|lake|town
5                  OH, Columbus, 5559 Orville Avenue     us|columbus|orville
6  2505, Tower 1, Oakwood, Runwal Greens, Mulund ...     india|tower|oakwood
7       1111 Church Street, Unit 2007, Nashville, TN     us|church|nashville
8              337 Oakland Avenue, Michigan City, IN     us|oakland|michigan
9  294 Meadowcreek Drive, Unit Unit 2, Village Of...  us|meadowcreek|village


In [109]:
#85
def build_address_first_two_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        key = row["block_address_first_two"]
        if key:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        key = row["block_address_first_two"]
        if key:
            index[key].append(row["entity_id"])

    return index


address_first_two_true_index = build_address_first_two_index(
    true_s2,
    true_s3
)

print(
    "Number of blocking keys:",
    len(address_first_two_true_index)
)

Number of blocking keys: 15709


In [110]:
#86
def generate_address_first_two_candidates(s1_row, index):
    key = s1_row["block_address_first_two"]

    if not key:
        return []

    return list(set(index.get(key, [])))


address_first_two_recall_rows = []

for _, row in s1.iterrows():
    true_ids = gt_sample.loc[
        gt_sample["source1_entity_id"] == row["entity_id"],
        "true_match_ids"
    ]

    if len(true_ids) == 0:
        continue

    true_ids = true_ids.iloc[0]

    candidates = set(
        generate_address_first_two_candidates(
            row,
            address_first_two_true_index
        )
    )

    recall = (
        len(true_ids & candidates) / len(true_ids)
        if len(true_ids) > 0
        else 1.0
    )

    address_first_two_recall_rows.append({
        "entity_id": row["entity_id"],
        "true_match_count": len(true_ids),
        "captured_match_count": len(true_ids & candidates),
        "recall": recall
    })


address_first_two_recall_df = pd.DataFrame(
    address_first_two_recall_rows
)

print(
    "Mean recall:",
    address_first_two_recall_df["recall"].mean()
)

print(
    "All true matches captured:",
    (address_first_two_recall_df["recall"] == 1.0).mean()
)

print(
    "At least one true match captured:",
    (
        address_first_two_recall_df["captured_match_count"] > 0
    ).mean()
)

Mean recall: 0.5648326984126983
All true matches captured: 0.2862
At least one true match captured: 0.739


In [111]:
#87
def compact_name_prefix_key(row, n=4):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return None

    name_compact = name.replace(" ", "")

    if len(name_compact) < n:
        return None

    return f"{country}|{name_compact[:n]}"

for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_compact_name_prefix"] = df.apply(
        compact_name_prefix_key,
        axis=1
    )

print(
    s1[
        [
            "business_name",
            "block_name_prefix",
            "block_compact_name_prefix"
        ]
    ].head(10)
)

                              business_name block_name_prefix  \
0                       Orelee's Barbershop           us|orel   
1                               Prime Money           us|prim   
2                             B+ Retail Inc           us|b re   
3                             Christ Chapel           us|chri   
4                   Prabhav Business Center        india|prab   
5                Custom Wealth Services LLC           us|cust   
6  Consulting Nyasa Nursing Private Limited        india|cons   
7                         Nexus Anchor Rain           us|nexu   
8                         Moore Bitwise Inc           us|moor   
9                Dermatology Green Medicine           us|derm   

  block_compact_name_prefix  
0                   us|orel  
1                   us|prim  
2                   us|bret  
3                   us|chri  
4                india|prab  
5                   us|cust  
6                india|cons  
7                   us|nexu  
8             

In [112]:
#88
def build_compact_name_prefix_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        key = row["block_compact_name_prefix"]
        if key:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        key = row["block_compact_name_prefix"]
        if key:
            index[key].append(row["entity_id"])

    return index


compact_name_prefix_true_index = build_compact_name_prefix_index(
    true_s2,
    true_s3
)

print(
    "Number of blocking keys:",
    len(compact_name_prefix_true_index)
)

Number of blocking keys: 6043


In [113]:
#89
def generate_compact_name_prefix_candidates(s1_row, index):
    key = s1_row["block_compact_name_prefix"]

    if not key:
        return []

    return list(set(index.get(key, [])))


compact_name_prefix_recall_rows = []

for _, row in s1.iterrows():
    true_ids = gt_sample.loc[
        gt_sample["source1_entity_id"] == row["entity_id"],
        "true_match_ids"
    ]

    if len(true_ids) == 0:
        continue

    true_ids = true_ids.iloc[0]

    candidates = set(
        generate_compact_name_prefix_candidates(
            row,
            compact_name_prefix_true_index
        )
    )

    recall = (
        len(true_ids & candidates) / len(true_ids)
        if len(true_ids) > 0
        else 1.0
    )

    compact_name_prefix_recall_rows.append({
        "entity_id": row["entity_id"],
        "true_match_count": len(true_ids),
        "captured_match_count": len(true_ids & candidates),
        "recall": recall
    })


compact_name_prefix_recall_df = pd.DataFrame(
    compact_name_prefix_recall_rows
)

print(
    "Mean recall:",
    compact_name_prefix_recall_df["recall"].mean()
)

print(
    "All true matches captured:",
    (compact_name_prefix_recall_df["recall"] == 1.0).mean()
)

print(
    "At least one true match captured:",
    (
        compact_name_prefix_recall_df["captured_match_count"] > 0
    ).mean()
)

Mean recall: 0.780388805916306
All true matches captured: 0.5039
At least one true match captured: 0.8991


In [114]:
#90
compact_name_prefix_sample_index = build_compact_name_prefix_index(
    s2,
    s3
)

print(
    "Number of blocking keys:",
    len(compact_name_prefix_sample_index)
)

Number of blocking keys: 7111


In [115]:
#91
compact_name_prefix_candidate_counts = []

for _, row in s1.iterrows():
    candidates = generate_compact_name_prefix_candidates(
        row,
        compact_name_prefix_sample_index
    )

    compact_name_prefix_candidate_counts.append(
        len(candidates)
    )


compact_name_prefix_candidate_counts = np.array(
    compact_name_prefix_candidate_counts
)

print(
    "Average candidates per S1:",
    compact_name_prefix_candidate_counts.mean()
)

print(
    "Maximum candidates per S1:",
    compact_name_prefix_candidate_counts.max()
)

print(
    "Minimum candidates per S1:",
    compact_name_prefix_candidate_counts.min()
)

print(
    "Std:",
    compact_name_prefix_candidate_counts.std()
)

print(
    "25%:",
    np.percentile(compact_name_prefix_candidate_counts, 25)
)

print(
    "Median:",
    np.percentile(compact_name_prefix_candidate_counts, 50)
)

print(
    "75%:",
    np.percentile(compact_name_prefix_candidate_counts, 75)
)

print(
    "S1 with at least one candidate:",
    (compact_name_prefix_candidate_counts > 0).sum()
)

Average candidates per S1: 13.1172
Maximum candidates per S1: 125
Minimum candidates per S1: 0
Std: 18.587147822083946
25%: 0.0
Median: 5.0
75%: 20.0
S1 with at least one candidate: 7431


In [116]:
#92
total_possible_candidates = len(s2) + len(s3)

compact_name_reduction_ratio = (
    1 - (
        compact_name_prefix_candidate_counts.mean()
        / total_possible_candidates
    )
)

print(
    "Total possible candidates per S1:",
    total_possible_candidates
)

print(
    "Average candidates per S1:",
    compact_name_prefix_candidate_counts.mean()
)

print(
    "Reduction ratio:",
    compact_name_reduction_ratio
)

print(
    "Reduction percentage:",
    compact_name_reduction_ratio * 100
)

Total possible candidates per S1: 20000
Average candidates per S1: 13.1172
Reduction ratio: 0.99934414
Reduction percentage: 99.934414


In [117]:
#93
blocking_results = pd.concat([
    blocking_results,
    pd.DataFrame([{
        "strategy": "Compact Name Prefix",
        "mean_recall": 0.7803888059,
        "all_matches_captured": 0.5039,
        "at_least_one_match": 0.8991,
        "avg_candidates": 13.1172,
        "max_candidates": 125,
        "reduction_ratio": 0.99934414
    }])
], ignore_index=True)

blocking_results

,strategy,mean_recall,all_matches_captured,at_least_one_match,avg_candidates,max_candidates,reduction_ratio
0,Name Prefix,0.768940,0.4510,0.9008,13.8694,176,0.999307
1,Address Prefix,0.478682,0.1768,0.7039,9.9810,375,0.999501
2,Name OR Address Prefix,0.880691,0.6530,0.9283,23.8334,429,0.998808
3,First Token,0.728741,0.3762,0.8926,10.1926,176,0.999490
4,Name OR Address OR First Token,0.883354,0.6587,0.9285,24.1028,429,0.998795
5,Address Token,0.949914,0.7904,0.9397,505.6911,3367,0.974715
6,Address Two-Token,0.794895,0.6014,0.8630,86.5395,948,0.995673
7,Compact Name Prefix,0.780389,0.5039,0.8991,13.1172,125,0.999344


In [118]:
#94
def first_last_name_token_key(row):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return None

    tokens = [
        token
        for token in name.split()
        if len(token) >= 2
    ]

    if not tokens:
        return None

    if len(tokens) == 1:
        return f"{country}|{tokens[0]}"

    return f"{country}|{tokens[0]}|{tokens[-1]}"

for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_first_last_name"] = df.apply(
        first_last_name_token_key,
        axis=1
    )

print(
    s1[
        [
            "business_name",
            "block_first_last_name"
        ]
    ].head(15)
)

                               business_name     block_first_last_name
0                        Orelee's Barbershop      us|orelee|barbershop
1                                Prime Money            us|prime|money
2                              B+ Retail Inc             us|retail|inc
3                              Christ Chapel          us|christ|chapel
4                    Prabhav Business Center      india|prabhav|center
5                 Custom Wealth Services LLC             us|custom|llc
6   Consulting Nyasa Nursing Private Limited  india|consulting|limited
7                          Nexus Anchor Rain             us|nexus|rain
8                          Moore Bitwise Inc              us|moore|inc
9                 Dermatology Green Medicine   us|dermatology|medicine
10                        Crystal Lending PC             us|crystal|pc
11                Dream Construction Limited       india|dream|limited
12                       Kelly Advisory, Inc              us|kelly|inc
13    

In [119]:
#95
def build_first_last_name_index(s2, s3):
    index = defaultdict(list)

    for _, row in s2.iterrows():
        key = row["block_first_last_name"]
        if key:
            index[key].append(row["entity_id"])

    for _, row in s3.iterrows():
        key = row["block_first_last_name"]
        if key:
            index[key].append(row["entity_id"])

    return index


first_last_name_true_index = build_first_last_name_index(
    true_s2,
    true_s3
)

print(
    "Number of blocking keys:",
    len(first_last_name_true_index)
)

Number of blocking keys: 20734


In [120]:
#96
def generate_first_last_name_candidates(s1_row, index):
    key = s1_row["block_first_last_name"]

    if not key:
        return []

    return list(set(index.get(key, [])))


first_last_name_recall_rows = []

for _, row in s1.iterrows():
    true_ids = gt_sample.loc[
        gt_sample["source1_entity_id"] == row["entity_id"],
        "true_match_ids"
    ]

    if len(true_ids) == 0:
        continue

    true_ids = true_ids.iloc[0]

    candidates = set(
        generate_first_last_name_candidates(
            row,
            first_last_name_true_index
        )
    )

    recall = (
        len(true_ids & candidates) / len(true_ids)
        if len(true_ids) > 0
        else 1.0
    )

    first_last_name_recall_rows.append({
        "entity_id": row["entity_id"],
        "true_match_count": len(true_ids),
        "captured_match_count": len(true_ids & candidates),
        "recall": recall
    })


first_last_name_recall_df = pd.DataFrame(
    first_last_name_recall_rows
)

print(
    "Mean recall:",
    first_last_name_recall_df["recall"].mean()
)

print(
    "All true matches captured:",
    (first_last_name_recall_df["recall"] == 1.0).mean()
)

print(
    "At least one true match captured:",
    (
        first_last_name_recall_df["captured_match_count"] > 0
    ).mean()
)

Mean recall: 0.3915560209235209
All true matches captured: 0.1182
At least one true match captured: 0.6961


In [121]:
#97
COMMON_NAME_TOKENS = {
    "inc",
    "llc",
    "ltd",
    "limited",
    "corp",
    "corporation",
    "company",
    "co",
    "plc",
    "private",
    "privatelimited",
    "pvt",
    "pvtltd",
    "group",
    "holdings",
    "holding",
    "international",
    "intl",
    "services",
    "service"
}


def name_token_keys_filtered(row):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return []

    tokens = [
        token
        for token in name.split()
        if len(token) >= 3
        and token not in COMMON_NAME_TOKENS
    ]

    tokens = list(dict.fromkeys(tokens))

    return [
        f"{country}|{token}"
        for token in tokens
    ]


for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_name_tokens"] = df.apply(
        name_token_keys_filtered,
        axis=1
    )

print(
    s1[
        [
            "business_name",
            "block_name_tokens"
        ]
    ].head(15)
)

                               business_name  \
0                        Orelee's Barbershop   
1                                Prime Money   
2                              B+ Retail Inc   
3                              Christ Chapel   
4                    Prabhav Business Center   
5                 Custom Wealth Services LLC   
6   Consulting Nyasa Nursing Private Limited   
7                          Nexus Anchor Rain   
8                          Moore Bitwise Inc   
9                 Dermatology Green Medicine   
10                        Crystal Lending PC   
11                Dream Construction Limited   
12                       Kelly Advisory, Inc   
13                                    Helios   
14                    Unified Choice Dynamix   

                                 block_name_tokens  
0                       [us|orelee, us|barbershop]  
1                             [us|prime, us|money]  
2                                      [us|retail]  
3                  

In [122]:
#98
# Block 98: Evaluate name-token recall on true entities

from collections import defaultdict

name_token_true_index = defaultdict(set)

for _, row in true_s2.iterrows():
    for key in row["block_name_tokens"]:
        name_token_true_index[key].add(row["entity_id"])

for _, row in true_s3.iterrows():
    for key in row["block_name_tokens"]:
        name_token_true_index[key].add(row["entity_id"])


recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():
    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    candidate_ids = set()

    for key in s1.loc[
        s1["entity_id"] == row["source1_entity_id"],
        "block_name_tokens"
    ].iloc[0]:
        candidate_ids.update(name_token_true_index.get(key, set()))

    captured = candidate_ids.intersection(true_ids)

    recall = len(captured) / len(true_ids)

    recall_values.append(recall)

    if recall == 1.0:
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1

    evaluated += 1


print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print("All true matches captured:", all_matches / evaluated)
print("At least one true match captured:", at_least_one / evaluated)

Evaluated S1: 9440
Mean recall: 0.8409234177428033
All true matches captured: 0.6201271186440678
At least one true match captured: 0.9711864406779661


In [123]:
#99
# Block 99: Candidate volume for name-token blocking

name_token_index = defaultdict(set)

for _, row in s2.iterrows():
    for key in row["block_name_tokens"]:
        name_token_index[key].add(row["entity_id"])

for _, row in s3.iterrows():
    for key in row["block_name_tokens"]:
        name_token_index[key].add(row["entity_id"])


candidate_counts = []

for _, row in s1.iterrows():
    candidate_ids = set()

    for key in row["block_name_tokens"]:
        candidate_ids.update(
            name_token_index.get(key, set())
        )

    candidate_counts.append(len(candidate_ids))


candidate_counts = np.array(candidate_counts)

total_possible = len(s2) + len(s3)

print("Index keys:", len(name_token_index))
print("Average candidates per S1:", candidate_counts.mean())
print("Max candidates per S1:", candidate_counts.max())
print("Min candidates per S1:", candidate_counts.min())
print("Std candidates per S1:", candidate_counts.std())
print("25th percentile:", np.percentile(candidate_counts, 25))
print("Median:", np.percentile(candidate_counts, 50))
print("75th percentile:", np.percentile(candidate_counts, 75))
print("S1 with at least one candidate:", np.sum(candidate_counts > 0))
print("Reduction ratio:", 1 - candidate_counts.sum() / (len(s1) * total_possible))

Index keys: 15112
Average candidates per S1: 113.0429
Max candidates per S1: 1367
Min candidates per S1: 0
Std candidates per S1: 154.58307300474397
25th percentile: 16.75
Median: 61.0
75th percentile: 115.0
S1 with at least one candidate: 9558
Reduction ratio: 0.994347855


In [124]:
#100
# Block 100: Unordered pair of informative name tokens

def name_two_token_pair_keys(row):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return []

    tokens = [
        token
        for token in name.split()
        if len(token) >= 3
        and token not in COMMON_NAME_TOKENS
    ]

    tokens = list(dict.fromkeys(tokens))

    if len(tokens) < 2:
        return []

    keys = []

    for i in range(len(tokens)):
        for j in range(i + 1, len(tokens)):
            pair = sorted([tokens[i], tokens[j]])
            keys.append(
                f"{country}|{pair[0]}|{pair[1]}"
            )

    return keys


for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_name_two_tokens"] = df.apply(
        name_two_token_pair_keys,
        axis=1
    )

print(
    s1[
        [
            "business_name",
            "block_name_two_tokens"
        ]
    ].head(15)
)

                               business_name  \
0                        Orelee's Barbershop   
1                                Prime Money   
2                              B+ Retail Inc   
3                              Christ Chapel   
4                    Prabhav Business Center   
5                 Custom Wealth Services LLC   
6   Consulting Nyasa Nursing Private Limited   
7                          Nexus Anchor Rain   
8                          Moore Bitwise Inc   
9                 Dermatology Green Medicine   
10                        Crystal Lending PC   
11                Dream Construction Limited   
12                       Kelly Advisory, Inc   
13                                    Helios   
14                    Unified Choice Dynamix   

                                block_name_two_tokens  
0                              [us|barbershop|orelee]  
1                                    [us|money|prime]  
2                                                  []  
3      

In [125]:
#101
# Block 101: Evaluate unordered name-token-pair recall

name_two_token_true_index = defaultdict(set)

for _, row in true_s2.iterrows():
    for key in row["block_name_two_tokens"]:
        name_two_token_true_index[key].add(row["entity_id"])

for _, row in true_s3.iterrows():
    for key in row["block_name_two_tokens"]:
        name_two_token_true_index[key].add(row["entity_id"])


recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():
    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    s1_row = s1.loc[
        s1["entity_id"] == row["source1_entity_id"]
    ].iloc[0]

    candidate_ids = set()

    for key in s1_row["block_name_two_tokens"]:
        candidate_ids.update(
            name_two_token_true_index.get(key, set())
        )

    captured = candidate_ids.intersection(true_ids)

    recall = len(captured) / len(true_ids)

    recall_values.append(recall)

    if recall == 1.0:
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1

    evaluated += 1


print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print("All true matches captured:", all_matches / evaluated)
print("At least one true match captured:", at_least_one / evaluated)

Evaluated S1: 9440
Mean recall: 0.684032604342456
All true matches captured: 0.41228813559322036
At least one true match captured: 0.8677966101694915


In [126]:
#102
# Block 102: Compare compact name-prefix lengths

def make_compact_name_prefix_key(row, n):
    name = row["name_normalized"]
    country = row["country_normalized"]

    if not name:
        return None

    name_compact = name.replace(" ", "")

    if len(name_compact) < n:
        return None

    return f"{country}|{name_compact[:n]}"


for n in [3, 4, 5]:
    for df in [s1, s2, s3, true_s2, true_s3]:
        df[f"block_name_prefix_{n}"] = df.apply(
            lambda row, n=n: make_compact_name_prefix_key(row, n),
            axis=1
        )

print(
    s1[
        [
            "business_name",
            "block_name_prefix_3",
            "block_name_prefix_4",
            "block_name_prefix_5"
        ]
    ].head(15)
)

                               business_name block_name_prefix_3  \
0                        Orelee's Barbershop              us|ore   
1                                Prime Money              us|pri   
2                              B+ Retail Inc              us|bre   
3                              Christ Chapel              us|chr   
4                    Prabhav Business Center           india|pra   
5                 Custom Wealth Services LLC              us|cus   
6   Consulting Nyasa Nursing Private Limited           india|con   
7                          Nexus Anchor Rain              us|nex   
8                          Moore Bitwise Inc              us|moo   
9                 Dermatology Green Medicine              us|der   
10                        Crystal Lending PC              us|cry   
11                Dream Construction Limited           india|dre   
12                       Kelly Advisory, Inc              us|kel   
13                                    Helios    

In [127]:
#103
# Block 103: Recall comparison for compact name prefixes

def evaluate_prefix_recall(prefix_col):
    prefix_true_index = defaultdict(set)

    for _, row in true_s2.iterrows():
        key = row[prefix_col]
        if key is not None:
            prefix_true_index[key].add(row["entity_id"])

    for _, row in true_s3.iterrows():
        key = row[prefix_col]
        if key is not None:
            prefix_true_index[key].add(row["entity_id"])

    recall_values = []
    all_matches = 0
    at_least_one = 0
    evaluated = 0

    for _, row in gt_sample.iterrows():
        true_ids = row["true_match_ids"]

        if not true_ids:
            continue

        s1_row = s1.loc[
            s1["entity_id"] == row["source1_entity_id"]
        ].iloc[0]

        key = s1_row[prefix_col]

        candidate_ids = prefix_true_index.get(key, set())

        captured = candidate_ids.intersection(true_ids)

        recall = len(captured) / len(true_ids)

        recall_values.append(recall)

        if recall == 1.0:
            all_matches += 1

        if len(captured) > 0:
            at_least_one += 1

        evaluated += 1

    return {
        "mean_recall": np.mean(recall_values),
        "all_matches_captured": all_matches / evaluated,
        "at_least_one_match": at_least_one / evaluated
    }


for n in [3, 4, 5]:
    result = evaluate_prefix_recall(
        f"block_name_prefix_{n}"
    )

    print(f"\nPrefix length: {n}")
    print("Mean recall:", result["mean_recall"])
    print("All true matches captured:", result["all_matches_captured"])
    print("At least one true match captured:", result["at_least_one_match"])


Prefix length: 3
Mean recall: 0.777221293595764
All true matches captured: 0.48983050847457626
At least one true match captured: 0.9557203389830509

Prefix length: 4
Mean recall: 0.7673610232164257
All true matches captured: 0.47447033898305085
At least one true match captured: 0.9524364406779661

Prefix length: 5
Mean recall: 0.7531813252072785
All true matches captured: 0.4538135593220339
At least one true match captured: 0.9475635593220338


In [128]:
#104
# Block 104: Evaluate 4- and 5-character compact name prefixes

for n in [4, 5]:
    result = evaluate_prefix_recall(
        f"block_name_prefix_{n}"
    )

    print(f"\nPrefix length: {n}")
    print("Mean recall:", result["mean_recall"])
    print("All true matches captured:", result["all_matches_captured"])
    print("At least one true match captured:", result["at_least_one_match"])


Prefix length: 4
Mean recall: 0.7673610232164257
All true matches captured: 0.47447033898305085
At least one true match captured: 0.9524364406779661

Prefix length: 5
Mean recall: 0.7531813252072785
All true matches captured: 0.4538135593220339
At least one true match captured: 0.9475635593220338


In [129]:
#105
# Block 105: Candidate volume for compact 3-character name prefix

name_prefix3_index = defaultdict(set)

for _, row in s2.iterrows():
    key = row["block_name_prefix_3"]
    if key is not None:
        name_prefix3_index[key].add(row["entity_id"])

for _, row in s3.iterrows():
    key = row["block_name_prefix_3"]
    if key is not None:
        name_prefix3_index[key].add(row["entity_id"])


candidate_counts_3 = []

for _, row in s1.iterrows():
    key = row["block_name_prefix_3"]

    if key is None:
        candidate_counts_3.append(0)
    else:
        candidate_counts_3.append(
            len(name_prefix3_index.get(key, set()))
        )


candidate_counts_3 = np.array(candidate_counts_3)

total_possible = len(s2) + len(s3)

print("Index keys:", len(name_prefix3_index))
print("Average candidates per S1:", candidate_counts_3.mean())
print("Max candidates per S1:", candidate_counts_3.max())
print("Min candidates per S1:", candidate_counts_3.min())
print("Std candidates per S1:", candidate_counts_3.std())
print("25th percentile:", np.percentile(candidate_counts_3, 25))
print("Median:", np.percentile(candidate_counts_3, 50))
print("75th percentile:", np.percentile(candidate_counts_3, 75))
print("S1 with at least one candidate:", np.sum(candidate_counts_3 > 0))
print(
    "Reduction ratio:",
    1 - candidate_counts_3.sum() / (len(s1) * total_possible)
)

Index keys: 4081
Average candidates per S1: 23.1242
Max candidates per S1: 210
Min candidates per S1: 0
Std candidates per S1: 27.128545378622864
25th percentile: 3.0
Median: 16.0
75th percentile: 33.0
S1 with at least one candidate: 8823
Reduction ratio: 0.99884379


In [130]:
#106
# Block 106: Compact 3-char Name Prefix OR Address Prefix

def generate_name3_or_address_candidates(row, blocking_index):
    candidate_ids = set()

    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        candidate_ids.update(
            blocking_index.get(name_key, set())
        )

    if address_key is not None:
        candidate_ids.update(
            blocking_index.get(address_key, set())
        )

    return candidate_ids

[c for c in s1.columns if "address" in c.lower()]

# Build combined actual-sample index

name3_address_index = defaultdict(set)

for _, row in s2.iterrows():
    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        name3_address_index[name_key].add(row["entity_id"])

    if address_key is not None:
        name3_address_index[address_key].add(row["entity_id"])


for _, row in s3.iterrows():
    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        name3_address_index[name_key].add(row["entity_id"])

    if address_key is not None:
        name3_address_index[address_key].add(row["entity_id"])


candidate_counts_name3_address = []

for _, row in s1.iterrows():
    candidates = generate_name3_or_address_candidates(
        row,
        name3_address_index
    )

    candidate_counts_name3_address.append(len(candidates))


candidate_counts_name3_address = np.array(
    candidate_counts_name3_address
)

total_possible = len(s2) + len(s3)

print("Index keys:", len(name3_address_index))
print(
    "Average candidates per S1:",
    candidate_counts_name3_address.mean()
)
print(
    "Max candidates per S1:",
    candidate_counts_name3_address.max()
)
print(
    "Min candidates per S1:",
    candidate_counts_name3_address.min()
)
print(
    "Std candidates per S1:",
    candidate_counts_name3_address.std()
)
print(
    "25th percentile:",
    np.percentile(candidate_counts_name3_address, 25)
)
print(
    "Median:",
    np.percentile(candidate_counts_name3_address, 50)
)
print(
    "75th percentile:",
    np.percentile(candidate_counts_name3_address, 75)
)
print(
    "S1 with at least one candidate:",
    np.sum(candidate_counts_name3_address > 0)
)
print(
    "Reduction ratio:",
    1 - candidate_counts_name3_address.sum()
    / (len(s1) * total_possible)
)

Index keys: 19338
Average candidates per S1: 33.0784
Max candidates per S1: 541
Min candidates per S1: 0
Std candidates per S1: 46.037305019299296
25th percentile: 5.0
Median: 19.0
75th percentile: 41.0
S1 with at least one candidate: 9141
Reduction ratio: 0.99834608


In [131]:
#107
# Block 107: True-match recall for Compact Name-3 OR Address Prefix

name3_true_index = defaultdict(set)
address_true_index = defaultdict(set)

# True S2
for _, row in true_s2.iterrows():
    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        name3_true_index[name_key].add(row["entity_id"])

    if address_key is not None:
        address_true_index[address_key].add(row["entity_id"])

# True S3
for _, row in true_s3.iterrows():
    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        name3_true_index[name_key].add(row["entity_id"])

    if address_key is not None:
        address_true_index[address_key].add(row["entity_id"])


recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    s1_row = s1.loc[
        s1["entity_id"] == row["source1_entity_id"]
    ].iloc[0]

    candidate_ids = set()

    name_key = s1_row["block_name_prefix_3"]
    address_key = s1_row["block_address_prefix"]

    if name_key is not None:
        candidate_ids.update(
            name3_true_index.get(name_key, set())
        )

    if address_key is not None:
        candidate_ids.update(
            address_true_index.get(address_key, set())
        )

    captured = candidate_ids.intersection(true_ids)

    recall = len(captured) / len(true_ids)

    recall_values.append(recall)

    if recall == 1.0:
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1

    evaluated += 1


print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print("All true matches captured:", all_matches / evaluated)
print("At least one true match captured:", at_least_one / evaluated)

Evaluated S1: 9440
Mean recall: 0.884562903551251
All true matches captured: 0.7007415254237288
At least one true match captured: 0.9837923728813559


In [134]:
#108

# Block 108: Name-3 OR Address Prefix OR Filtered Address Token

address_token_true_index = defaultdict(set)

for _, row in true_s2.iterrows():
    for key in row["block_address_tokens_filtered"]:
        address_token_true_index[key].add(row["entity_id"])

for _, row in true_s3.iterrows():
    for key in row["block_address_tokens_filtered"]:
        address_token_true_index[key].add(row["entity_id"])


recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    s1_row = s1.loc[
        s1["entity_id"] == row["source1_entity_id"]
    ].iloc[0]

    candidate_ids = set()

    # Rule 1: compact 3-character name prefix
    name_key = s1_row["block_name_prefix_3"]

    if name_key is not None:
        candidate_ids.update(
            name3_true_index.get(name_key, set())
        )

    # Rule 2: address prefix
    address_key = s1_row["block_address_prefix"]

    if address_key is not None:
        candidate_ids.update(
            address_true_index.get(address_key, set())
        )

    # Rule 3: filtered address tokens
    for key in s1_row["block_address_tokens_filtered"]:
        candidate_ids.update(
            address_token_true_index.get(key, set())
        )

    captured = candidate_ids.intersection(true_ids)

    recall = len(captured) / len(true_ids)

    recall_values.append(recall)

    if recall == 1.0:
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1

    evaluated += 1


print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print("All true matches captured:", all_matches / evaluated)
print("At least one true match captured:", at_least_one / evaluated)
      

Evaluated S1: 9440
Mean recall: 0.9976960586494484
All true matches captured: 0.9907838983050847
At least one true match captured: 1.0


In [135]:
#109
# Block 109: Candidate volume for
# Name-3 OR Address Prefix OR Filtered Address Token

combined_index = defaultdict(set)

for _, row in s2.iterrows():

    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        combined_index[("name", name_key)].add(row["entity_id"])

    if address_key is not None:
        combined_index[("address", address_key)].add(row["entity_id"])

    for key in row["block_address_tokens_filtered"]:
        combined_index[("address_token", key)].add(row["entity_id"])


for _, row in s3.iterrows():

    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        combined_index[("name", name_key)].add(row["entity_id"])

    if address_key is not None:
        combined_index[("address", address_key)].add(row["entity_id"])

    for key in row["block_address_tokens_filtered"]:
        combined_index[("address_token", key)].add(row["entity_id"])


candidate_counts_combined = []

for _, row in s1.iterrows():

    candidate_ids = set()

    name_key = row["block_name_prefix_3"]
    address_key = row["block_address_prefix"]

    if name_key is not None:
        candidate_ids.update(
            combined_index.get(
                ("name", name_key), set()
            )
        )

    if address_key is not None:
        candidate_ids.update(
            combined_index.get(
                ("address", address_key), set()
            )
        )

    for key in row["block_address_tokens_filtered"]:
        candidate_ids.update(
            combined_index.get(
                ("address_token", key), set()
            )
        )

    candidate_counts_combined.append(
        len(candidate_ids)
    )


candidate_counts_combined = np.array(
    candidate_counts_combined
)

total_possible = len(s2) + len(s3)

print("Index keys:", len(combined_index))
print(
    "Average candidates per S1:",
    candidate_counts_combined.mean()
)
print(
    "Max candidates per S1:",
    candidate_counts_combined.max()
)
print(
    "Min candidates per S1:",
    candidate_counts_combined.min()
)
print(
    "Std candidates per S1:",
    candidate_counts_combined.std()
)
print(
    "25th percentile:",
    np.percentile(candidate_counts_combined, 25)
)
print(
    "Median:",
    np.percentile(candidate_counts_combined, 50)
)
print(
    "75th percentile:",
    np.percentile(candidate_counts_combined, 75)
)
print(
    "S1 with at least one candidate:",
    np.sum(candidate_counts_combined > 0)
)
print(
    "Reduction ratio:",
    1 - candidate_counts_combined.sum()
    / (len(s1) * total_possible)
)

Index keys: 40748
Average candidates per S1: 530.2494
Max candidates per S1: 3369
Min candidates per S1: 0
Std candidates per S1: 631.3792296232431
25th percentile: 55.0
Median: 193.5
75th percentile: 879.25
S1 with at least one candidate: 9988
Reduction ratio: 0.97348753


In [136]:
#110
# Block 110: Address token frequency analysis

from collections import Counter

address_token_frequency = Counter()

for df in [s2, s3]:
    for tokens in df["block_address_tokens_filtered"]:
        address_token_frequency.update(tokens)

print("Unique address tokens:", len(address_token_frequency))

print("\nMost common tokens:")
for token, count in address_token_frequency.most_common(20):
    print(token, "->", count)

print("\nLeast common example tokens:")
for token, count in address_token_frequency.most_common()[-20:]:
    print(token, "->", count)

Unique address tokens: 21410

Most common tokens:
india|nagar -> 1086
india|delhi -> 992
us|city -> 781
india|new -> 723
india|mumbai -> 718
india|maharashtra -> 707
india|plot -> 695
india|west -> 583
us|new -> 579
us|texas -> 534
india|pradesh -> 494
india|city -> 491
us|north -> 484
india|bangalore -> 460
us|york -> 446
india|door -> 431
us|carolina -> 411
india|near -> 397
india|sector -> 378
india|colony -> 366

Least common example tokens:
india|frgp -> 1
us|trellis -> 1
us|whittington -> 1
india|vaddagere -> 1
india|koratagere -> 1
us|polaris -> 1
us|waden -> 1
us|tenbrouck -> 1
us|bernadette -> 1
india|administrative -> 1
india|fertilizernagar -> 1
us|drummond -> 1
india|kayper -> 1
india|pitchampalayam -> 1
india|tapookara -> 1
india|meadow -> 1
india|navadweepa -> 1
india|rajgruhi -> 1
india|track -> 1
india|maduvinkarai -> 1


In [137]:
#111
# Block 111: Moderately selective address-token keys
#
# Keep tokens occurring between 2 and 50 times
# across the S2 + S3 development sample.

MIN_TOKEN_FREQ = 2
MAX_TOKEN_FREQ = 50


def address_tokens_frequency_filtered(row):
    tokens = row["block_address_tokens_filtered"]

    return [
        token
        for token in tokens
        if MIN_TOKEN_FREQ
        <= address_token_frequency[token]
        <= MAX_TOKEN_FREQ
    ]


for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_address_tokens_freq_filtered"] = df.apply(
        address_tokens_frequency_filtered,
        axis=1
    )


print(
    s1[
        [
            "business_address",
            "block_address_tokens_filtered",
            "block_address_tokens_freq_filtered"
        ]
    ].head(20)
)

                                     business_address  \
0              1795 Westchester Drive, High Point, NC   
1                     17560 Ellis Road, Tahlequah, OK   
2                 1712 Montebello Avenue, Phoenix, AZ   
3   2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD   
4   797, Lake Town Block A, Kolkata, Howrah, West ...   
5                   OH, Columbus, 5559 Orville Avenue   
6   2505, Tower 1, Oakwood, Runwal Greens, Mulund ...   
7        1111 Church Street, Unit 2007, Nashville, TN   
8               337 Oakland Avenue, Michigan City, IN   
9   294 Meadowcreek Drive, Unit Unit 2, Village Of...   
10            11643 Prosperity Road, South Jordan, UT   
11  H.No.16-11-23/37/A, 2Nd Floor, Flat No.207, Sa...   
12                         301 1st Street, Chokio, MN   
13                 66 Edgewood Street, Bridgeport, CT   
14  Unit BUILDING 3030, MD, 2701 Eastern Boulevard...   
15       601 Oleander Circle, Virginia Beach City, VA   
16  303, 3Rd Floor Sakar 5 B/H 

In [138]:
#112
# Block 112: Evaluate frequency-filtered address tokens

from collections import defaultdict

def build_multi_key_index(df, key_column):
    index = defaultdict(list)

    for _, row in df.iterrows():
        for key in row[key_column]:
            index[key].append(row["entity_id"])

    return index


def generate_from_multi_key_index(df, index, key_column):
    candidates = []

    for _, row in df.iterrows():
        ids = set()

        for key in row[key_column]:
            ids.update(index.get(key, []))

        candidates.append(ids)

    return candidates


# ---------------------------------------------------------
# 1. Candidate volume on actual 10k S2 + S3 sample
# ---------------------------------------------------------

freq_index_sample = build_multi_key_index(
    pd.concat([s2, s3], ignore_index=True),
    "block_address_tokens_freq_filtered"
)

freq_candidates = generate_from_multi_key_index(
    s1,
    freq_index_sample,
    "block_address_tokens_freq_filtered"
)

freq_candidate_counts = np.array(
    [len(x) for x in freq_candidates]
)

print("=== Candidate Volume ===")
print("Index keys:", len(freq_index_sample))
print("Average candidates/S1:", freq_candidate_counts.mean())
print("Max candidates/S1:", freq_candidate_counts.max())
print("Min candidates/S1:", freq_candidate_counts.min())
print("Std:", freq_candidate_counts.std())
print("25%:", np.percentile(freq_candidate_counts, 25))
print("Median:", np.percentile(freq_candidate_counts, 50))
print("75%:", np.percentile(freq_candidate_counts, 75))
print(
    "S1 with at least one candidate:",
    np.sum(freq_candidate_counts > 0)
)

total_possible = len(s2) + len(s3)

reduction_ratio = 1 - (
    freq_candidate_counts.sum()
    / (len(s1) * total_possible)
)

print("Reduction ratio:", reduction_ratio)


# ---------------------------------------------------------
# 2. Exact blocking recall using true-only entities
# ---------------------------------------------------------

freq_index_true = build_multi_key_index(
    pd.concat([true_s2, true_s3], ignore_index=True),
    "block_address_tokens_freq_filtered"
)

freq_true_candidates = generate_from_multi_key_index(
    s1,
    freq_index_true,
    "block_address_tokens_freq_filtered"
)

recall_values = []
all_matches_captured = 0
at_least_one_captured = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    candidate_ids = freq_true_candidates[
        s1.index[s1["entity_id"] == row["source1_entity_id"]][0]
    ]

    captured = true_ids.intersection(candidate_ids)

    recall = len(captured) / len(true_ids)
    recall_values.append(recall)

    if len(captured) == len(true_ids):
        all_matches_captured += 1

    if len(captured) > 0:
        at_least_one_captured += 1


print("\n=== Blocking Recall ===")
print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print(
    "All true matches captured:",
    all_matches_captured / evaluated
)
print(
    "At least one true match captured:",
    at_least_one_captured / evaluated
)

=== Candidate Volume ===
Index keys: 6406
Average candidates/S1: 24.2652
Max candidates/S1: 200
Min candidates/S1: 0
Std: 23.87690660366204
25%: 5.0
Median: 17.0
75%: 37.0
S1 with at least one candidate: 8818
Reduction ratio: 0.99878674

=== Blocking Recall ===
Evaluated S1: 9440
Mean recall: 0.7700108034032822
All true matches captured: 0.6119703389830509
At least one true match captured: 0.8632415254237288


In [139]:
#113
# Block 113: Identify S1 records that need an address-token fallback

from collections import defaultdict

# Build indexes for the current strong baseline
baseline_index = defaultdict(list)

for df in [s2, s3]:
    for _, row in df.iterrows():
        for key in [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]:
            if pd.notna(key) and key is not None:
                baseline_index[key].append(row["entity_id"])


# Generate baseline candidates
baseline_candidates = []

for _, row in s1.iterrows():
    candidate_ids = set()

    for key in [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]:
        if pd.notna(key) and key is not None:
            candidate_ids.update(baseline_index.get(key, []))

    baseline_candidates.append(candidate_ids)


baseline_counts = np.array(
    [len(x) for x in baseline_candidates]
)

print("=== Baseline coverage ===")
print(
    "S1 with >=1 candidate:",
    np.sum(baseline_counts > 0)
)
print(
    "S1 with 0 candidates:",
    np.sum(baseline_counts == 0)
)

print("\nCandidate count distribution for uncovered S1:")
print(
    "Uncovered percentage:",
    np.mean(baseline_counts == 0)
)

# Show a few uncovered examples
uncovered_indices = np.where(baseline_counts == 0)[0][:20]

print("\n=== Example uncovered S1 records ===")

print(
    s1.loc[
        uncovered_indices,
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
            "block_name_prefix_3",
            "block_address_prefix"
        ]
    ].to_string(index=False)
)

=== Baseline coverage ===
S1 with >=1 candidate: 9141
S1 with 0 candidates: 859

Candidate count distribution for uncovered S1:
Uncovered percentage: 0.0859

=== Example uncovered S1 records ===
   entity_id                          business_name                                                                            business_address country block_name_prefix_3 block_address_prefix
S1-252242682               Aggie E. Nagle, L.C.S.W.                                                             11900 54th Avenue, Plymouth, MN      US              us|agg            us|11900 
S1-791867209               WG Harbor Worldwide P.C.                                                               101 Oak Street, Lohrville, IA      US              us|wgh            us|101 oa
S1-666876590                     Ss Systems Limited                    A-1, Malad (West), Mumbai, Maharashtra, 401, Rahul Apartment, Marve Road   India           india|sss         india|a 1 ma
S1-960158214     Tinoco Institute

In [140]:
#114
# Block 114: Address-token fallback for baseline-uncovered S1 records

# Build address-token index from the actual S2 + S3 development sample
fallback_address_index = defaultdict(list)

for df in [s2, s3]:
    for _, row in df.iterrows():
        for token in row["block_address_tokens_filtered"]:
            fallback_address_index[token].append(row["entity_id"])


# Apply fallback ONLY to S1 records with zero baseline candidates
fallback_candidates = []

for i, row in s1.iterrows():

    # Already covered by Name-3 OR Address-prefix
    if baseline_counts[i] > 0:
        fallback_candidates.append(set())
        continue

    candidate_ids = set()

    for token in row["block_address_tokens_filtered"]:
        candidate_ids.update(
            fallback_address_index.get(token, [])
        )

    fallback_candidates.append(candidate_ids)


fallback_counts = np.array(
    [len(x) for x in fallback_candidates]
)


# ---------------------------------------------------------
# Coverage from fallback
# ---------------------------------------------------------

print("=== Fallback candidate volume ===")
print(
    "Uncovered S1:",
    np.sum(baseline_counts == 0)
)
print(
    "Fallback S1 with candidates:",
    np.sum(fallback_counts > 0)
)
print(
    "Fallback average candidates among uncovered:",
    fallback_counts[baseline_counts == 0].mean()
)
print(
    "Fallback max candidates:",
    fallback_counts.max()
)


# ---------------------------------------------------------
# Combined candidate volume
# ---------------------------------------------------------

combined_counts = baseline_counts + fallback_counts

print("\n=== Combined candidate volume ===")
print(
    "Average candidates/S1:",
    combined_counts.mean()
)
print(
    "Max candidates/S1:",
    combined_counts.max()
)
print(
    "S1 with >=1 candidate:",
    np.sum(combined_counts > 0)
)

total_possible = len(s2) + len(s3)

combined_reduction = 1 - (
    combined_counts.sum()
    / (len(s1) * total_possible)
)

print(
    "Reduction ratio:",
    combined_reduction
)


# ---------------------------------------------------------
# Show fallback examples
# ---------------------------------------------------------

fallback_indices = np.where(
    (baseline_counts == 0) &
    (fallback_counts > 0)
)[0][:20]

print("\n=== Example fallback-covered S1 ===")

print(
    s1.loc[
        fallback_indices,
        [
            "entity_id",
            "business_name",
            "business_address",
            "block_address_tokens_filtered"
        ]
    ].to_string(index=False)
)

=== Fallback candidate volume ===
Uncovered S1: 859
Fallback S1 with candidates: 847
Fallback average candidates among uncovered: 451.6682188591385
Fallback max candidates: 2417

=== Combined candidate volume ===
Average candidates/S1: 71.8767
Max candidates/S1: 2417
S1 with >=1 candidate: 9988
Reduction ratio: 0.996406165

=== Example fallback-covered S1 ===
   entity_id                          business_name                                                                            business_address                                                                                          block_address_tokens_filtered
S1-252242682               Aggie E. Nagle, L.C.S.W.                                                             11900 54th Avenue, Plymouth, MN                                                                                                 [us|54th, us|plymouth]
S1-791867209               WG Harbor Worldwide P.C.                                                             

In [141]:
#115
# Block 115: Two-token address fallback for baseline-uncovered S1

two_token_fallback_index = defaultdict(list)

for df in [s2, s3]:
    for _, row in df.iterrows():
        for key in row["block_address_two_tokens"]:
            two_token_fallback_index[key].append(row["entity_id"])


two_token_fallback_candidates = []

for i, row in s1.iterrows():

    # Only apply to baseline-uncovered records
    if baseline_counts[i] > 0:
        two_token_fallback_candidates.append(set())
        continue

    candidate_ids = set()

    for key in row["block_address_two_tokens"]:
        candidate_ids.update(
            two_token_fallback_index.get(key, [])
        )

    two_token_fallback_candidates.append(candidate_ids)


two_token_fallback_counts = np.array(
    [len(x) for x in two_token_fallback_candidates]
)


print("=== Two-token fallback candidate volume ===")
print(
    "Uncovered S1:",
    np.sum(baseline_counts == 0)
)
print(
    "Fallback S1 with candidates:",
    np.sum(two_token_fallback_counts > 0)
)
print(
    "Fallback average candidates among uncovered:",
    two_token_fallback_counts[
        baseline_counts == 0
    ].mean()
)
print(
    "Fallback max candidates:",
    two_token_fallback_counts.max()
)

combined_two_counts = (
    baseline_counts +
    two_token_fallback_counts
)

print("\n=== Combined candidate volume ===")
print(
    "Average candidates/S1:",
    combined_two_counts.mean()
)
print(
    "Max candidates/S1:",
    combined_two_counts.max()
)
print(
    "S1 with >=1 candidate:",
    np.sum(combined_two_counts > 0)
)

total_possible = len(s2) + len(s3)

combined_two_reduction = 1 - (
    combined_two_counts.sum()
    / (len(s1) * total_possible)
)

print(
    "Reduction ratio:",
    combined_two_reduction
)

=== Two-token fallback candidate volume ===
Uncovered S1: 859
Fallback S1 with candidates: 472
Fallback average candidates among uncovered: 80.0011641443539
Fallback max candidates: 705

=== Combined candidate volume ===
Average candidates/S1: 39.9505
Max candidates/S1: 705
S1 with >=1 candidate: 9613
Reduction ratio: 0.9980024750000001


In [142]:
#116
# Block 116: Exact recall for
# Name-3 OR Address-prefix + two-token address fallback

# Build true-entity index for the two-token fallback
two_token_true_index = defaultdict(list)

for df in [true_s2, true_s3]:
    for _, row in df.iterrows():
        for key in row["block_address_two_tokens"]:
            two_token_true_index[key].append(row["entity_id"])


# Generate true candidates from the fallback
two_token_true_candidates = []

for i, row in s1.iterrows():

    # Only fallback for baseline-uncovered S1
    if baseline_counts[i] > 0:
        two_token_true_candidates.append(set())
        continue

    candidate_ids = set()

    for key in row["block_address_two_tokens"]:
        candidate_ids.update(
            two_token_true_index.get(key, [])
        )

    two_token_true_candidates.append(candidate_ids)


# ---------------------------------------------------------
# Evaluate combined baseline + fallback recall
# ---------------------------------------------------------

recall_values = []
all_matches_captured = 0
at_least_one_captured = 0
evaluated = 0

# Also measure baseline alone for direct comparison
baseline_recall_values = []
baseline_all = 0
baseline_at_least_one = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    s1_position = s1.index[
        s1["entity_id"] == row["source1_entity_id"]
    ][0]

    # Baseline candidates
    baseline_ids = baseline_candidates[s1_position]

    # Fallback candidates
    fallback_ids = two_token_true_candidates[s1_position]

    # Combined candidates
    combined_ids = baseline_ids.union(fallback_ids)

    # ----- baseline only -----
    baseline_captured = true_ids.intersection(
        baseline_ids
    )

    baseline_recall = (
        len(baseline_captured) / len(true_ids)
    )

    baseline_recall_values.append(baseline_recall)

    if len(baseline_captured) == len(true_ids):
        baseline_all += 1

    if len(baseline_captured) > 0:
        baseline_at_least_one += 1

    # ----- combined -----
    captured = true_ids.intersection(
        combined_ids
    )

    recall = len(captured) / len(true_ids)

    recall_values.append(recall)

    if len(captured) == len(true_ids):
        all_matches_captured += 1

    if len(captured) > 0:
        at_least_one_captured += 1


print("=== Baseline recall ===")
print(
    "Mean recall:",
    np.mean(baseline_recall_values)
)
print(
    "All true matches captured:",
    baseline_all / evaluated
)
print(
    "At least one true match captured:",
    baseline_at_least_one / evaluated
)


print("\n=== Two-token fallback combined recall ===")
print(
    "Evaluated S1:",
    evaluated
)
print(
    "Mean recall:",
    np.mean(recall_values)
)
print(
    "All true matches captured:",
    all_matches_captured / evaluated
)
print(
    "At least one true match captured:",
    at_least_one_captured / evaluated
)


print("\n=== Recall improvement ===")
print(
    "Mean recall improvement:",
    np.mean(recall_values)
    - np.mean(baseline_recall_values)
)
print(
    "All-match improvement:",
    (all_matches_captured / evaluated)
    - (baseline_all / evaluated)
)
print(
    "At-least-one improvement:",
    (at_least_one_captured / evaluated)
    - (baseline_at_least_one / evaluated)
)

=== Baseline recall ===
Mean recall: 0.0018152239709443097
All true matches captured: 0.0001059322033898305
At least one true match captured: 0.00625

=== Two-token fallback combined recall ===
Evaluated S1: 9440
Mean recall: 0.0681035193032015
All true matches captured: 0.04766949152542373
At least one true match captured: 0.08347457627118644

=== Recall improvement ===
Mean recall improvement: 0.06628829533225719
All-match improvement: 0.0475635593220339
At-least-one improvement: 0.07722457627118644


In [143]:
#117
# Block 117: Corrected exact recall comparison
#
# IMPORTANT:
# Both baseline and fallback are evaluated against the
# true-only S2/S3 entities.

# ---------------------------------------------------------
# 1. Build true-only baseline index
# ---------------------------------------------------------

true_baseline_index = defaultdict(list)

for df in [true_s2, true_s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        for key in keys:
            if pd.notna(key) and key is not None:
                true_baseline_index[key].append(
                    row["entity_id"]
                )


# ---------------------------------------------------------
# 2. Generate true-only baseline candidates
# ---------------------------------------------------------

true_baseline_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                true_baseline_index.get(key, [])
            )

    true_baseline_candidates.append(candidate_ids)


# ---------------------------------------------------------
# 3. Evaluate baseline + two-token fallback
# ---------------------------------------------------------

recall_baseline = []
recall_combined = []

baseline_all = 0
baseline_at_least_one = 0

combined_all = 0
combined_at_least_one = 0

evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    s1_position = s1.index[
        s1["entity_id"] == row["source1_entity_id"]
    ][0]

    baseline_ids = true_baseline_candidates[
        s1_position
    ]

    fallback_ids = two_token_true_candidates[
        s1_position
    ]

    combined_ids = baseline_ids.union(
        fallback_ids
    )

    # Baseline
    baseline_captured = true_ids.intersection(
        baseline_ids
    )

    baseline_recall_value = (
        len(baseline_captured) / len(true_ids)
    )

    recall_baseline.append(
        baseline_recall_value
    )

    if len(baseline_captured) == len(true_ids):
        baseline_all += 1

    if len(baseline_captured) > 0:
        baseline_at_least_one += 1

    # Combined
    combined_captured = true_ids.intersection(
        combined_ids
    )

    combined_recall_value = (
        len(combined_captured) / len(true_ids)
    )

    recall_combined.append(
        combined_recall_value
    )

    if len(combined_captured) == len(true_ids):
        combined_all += 1

    if len(combined_captured) > 0:
        combined_at_least_one += 1


# ---------------------------------------------------------
# 4. Results
# ---------------------------------------------------------

baseline_mean = np.mean(recall_baseline)
combined_mean = np.mean(recall_combined)

baseline_all_rate = baseline_all / evaluated
combined_all_rate = combined_all / evaluated

baseline_one_rate = baseline_at_least_one / evaluated
combined_one_rate = combined_at_least_one / evaluated


print("=== Corrected baseline recall ===")
print("Evaluated S1:", evaluated)
print("Mean recall:", baseline_mean)
print("All true matches captured:", baseline_all_rate)
print("At least one true match captured:", baseline_one_rate)


print("\n=== Baseline + two-token fallback ===")
print("Mean recall:", combined_mean)
print("All true matches captured:", combined_all_rate)
print(
    "At least one true match captured:",
    combined_one_rate
)


print("\n=== Improvement ===")
print(
    "Mean recall improvement:",
    combined_mean - baseline_mean
)
print(
    "All-match improvement:",
    combined_all_rate - baseline_all_rate
)
print(
    "At-least-one improvement:",
    combined_one_rate - baseline_one_rate
)

=== Corrected baseline recall ===
Evaluated S1: 9440
Mean recall: 0.884562903551251
All true matches captured: 0.7007415254237288
At least one true match captured: 0.9837923728813559

=== Baseline + two-token fallback ===
Mean recall: 0.8906633794054345
All true matches captured: 0.7185381355932203
At least one true match captured: 0.9840042372881356

=== Improvement ===
Mean recall improvement: 0.006100475854183496
All-match improvement: 0.017796610169491522
At-least-one improvement: 0.00021186440677967155


In [144]:
#118
# Block 118: Name-3 + Address-prefix + frequency-filtered address tokens

combined_freq_index = defaultdict(list)

for df in [s2, s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_filtered"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                combined_freq_index[key].append(
                    row["entity_id"]
                )


# Generate candidates
combined_freq_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_filtered"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                combined_freq_index.get(key, [])
            )

    combined_freq_candidates.append(candidate_ids)


# Candidate volume
combined_freq_counts = np.array(
    [len(x) for x in combined_freq_candidates]
)

print("=== Candidate Volume ===")
print("Index keys:", len(combined_freq_index))
print(
    "Average candidates/S1:",
    combined_freq_counts.mean()
)
print(
    "Max candidates/S1:",
    combined_freq_counts.max()
)
print(
    "Min candidates/S1:",
    combined_freq_counts.min()
)
print(
    "Median:",
    np.percentile(combined_freq_counts, 50)
)
print(
    "75%:",
    np.percentile(combined_freq_counts, 75)
)
print(
    "S1 with >=1 candidate:",
    np.sum(combined_freq_counts > 0)
)

total_possible = len(s2) + len(s3)

reduction_ratio = 1 - (
    combined_freq_counts.sum()
    / (len(s1) * total_possible)
)

print(
    "Reduction ratio:",
    reduction_ratio
)

=== Candidate Volume ===
Index keys: 25343
Average candidates/S1: 59.2115
Max candidates/S1: 549
Min candidates/S1: 0
Median: 46.0
75%: 76.0
S1 with >=1 candidate: 9902
Reduction ratio: 0.997039425


In [145]:
#119
# Block 119: Exact recall for
# Name-3 OR Address-prefix OR frequency-filtered address token

true_combined_freq_index = defaultdict(list)

for df in [true_s2, true_s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_filtered"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                true_combined_freq_index[key].append(
                    row["entity_id"]
                )


true_combined_freq_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_filtered"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                true_combined_freq_index.get(key, [])
            )

    true_combined_freq_candidates.append(candidate_ids)


# Evaluate recall
recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    s1_position = s1.index[
        s1["entity_id"] == row["source1_entity_id"]
    ][0]

    captured = true_ids.intersection(
        true_combined_freq_candidates[s1_position]
    )

    recall_values.append(
        len(captured) / len(true_ids)
    )

    if len(captured) == len(true_ids):
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1


print("=== Exact Recall ===")
print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print(
    "All true matches captured:",
    all_matches / evaluated
)
print(
    "At least one true match captured:",
    at_least_one / evaluated
)

=== Exact Recall ===
Evaluated S1: 9440
Mean recall: 0.9727297719935433
All true matches captured: 0.9204449152542373
At least one true match captured: 0.9972457627118644


In [146]:
#120
# Block 120: Frequency 2-20 address tokens

MIN_TOKEN_FREQ_20 = 2
MAX_TOKEN_FREQ_20 = 20


def address_tokens_freq_2_20(row):
    tokens = row["block_address_tokens_filtered"]

    return [
        token
        for token in tokens
        if MIN_TOKEN_FREQ_20
        <= address_token_frequency[token]
        <= MAX_TOKEN_FREQ_20
    ]


for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_address_tokens_freq_2_20"] = df.apply(
        address_tokens_freq_2_20,
        axis=1
    )


print("=== Examples ===")

print(
    s1[
        [
            "business_address",
            "block_address_tokens_filtered",
            "block_address_tokens_freq_2_20"
        ]
    ].head(20).to_string(index=False)
)

=== Examples ===
                                                                                                         business_address                                                                                                                                                                            block_address_tokens_filtered                            block_address_tokens_freq_2_20
                                                                                   1795 Westchester Drive, High Point, NC                                                                                                                                                                      [us|high, us|westchester, us|point]                                          [us|westchester]
                                                                                          17560 Ellis Road, Tahlequah, OK                                                                                                    

In [147]:
#121
# Block 121: Candidate volume for address-token frequency 2-20

freq_2_20_index = defaultdict(list)

for df in [s2, s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_2_20"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                freq_2_20_index[key].append(
                    row["entity_id"]
                )


freq_2_20_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_2_20"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                freq_2_20_index.get(key, [])
            )

    freq_2_20_candidates.append(candidate_ids)


freq_2_20_counts = np.array(
    [len(x) for x in freq_2_20_candidates]
)

print("=== Candidate Volume: Frequency 2-20 ===")
print("Index keys:", len(freq_2_20_index))
print(
    "Average candidates/S1:",
    freq_2_20_counts.mean()
)
print(
    "Max candidates/S1:",
    freq_2_20_counts.max()
)
print(
    "Min candidates/S1:",
    freq_2_20_counts.min()
)
print(
    "25%:",
    np.percentile(freq_2_20_counts, 25)
)
print(
    "Median:",
    np.percentile(freq_2_20_counts, 50)
)
print(
    "75%:",
    np.percentile(freq_2_20_counts, 75)
)
print(
    "S1 with >=1 candidate:",
    np.sum(freq_2_20_counts > 0)
)

total_possible = len(s2) + len(s3)

reduction_ratio_2_20 = 1 - (
    freq_2_20_counts.sum()
    / (len(s1) * total_possible)
)

print(
    "Reduction ratio:",
    reduction_ratio_2_20
)


=== Candidate Volume: Frequency 2-20 ===
Index keys: 25135
Average candidates/S1: 44.077
Max candidates/S1: 549
Min candidates/S1: 0
25%: 16.0
Median: 31.0
75%: 54.0
S1 with >=1 candidate: 9817
Reduction ratio: 0.99779615


In [148]:
# Block 122: Exact recall for
# Name-3 OR Address-prefix OR address-token frequency 2-20

true_freq_2_20_index = defaultdict(list)

for df in [true_s2, true_s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_2_20"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                true_freq_2_20_index[key].append(
                    row["entity_id"]
                )


true_freq_2_20_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_2_20"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                true_freq_2_20_index.get(key, [])
            )

    true_freq_2_20_candidates.append(candidate_ids)


# Evaluate
recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    s1_position = s1.index[
        s1["entity_id"] == row["source1_entity_id"]
    ][0]

    captured = true_ids.intersection(
        true_freq_2_20_candidates[s1_position]
    )

    recall_values.append(
        len(captured) / len(true_ids)
    )

    if len(captured) == len(true_ids):
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1


print("=== Exact Recall: Frequency 2-20 ===")
print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print(
    "All true matches captured:",
    all_matches / evaluated
)
print(
    "At least one true match captured:",
    at_least_one / evaluated
)

=== Exact Recall: Frequency 2-20 ===
Evaluated S1: 9440
Mean recall: 0.959822689669088
All true matches captured: 0.886228813559322
At least one true match captured: 0.9961864406779661


In [149]:
# Block 123: Address-token frequency 2-30

MIN_TOKEN_FREQ_30 = 2
MAX_TOKEN_FREQ_30 = 30

def address_tokens_freq_2_30(row):
    tokens = row["block_address_tokens_filtered"]
    return [
        token
        for token in tokens
        if MIN_TOKEN_FREQ_30 <= address_token_frequency[token] <= MAX_TOKEN_FREQ_30
    ]

for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_address_tokens_freq_2_30"] = df.apply(
        address_tokens_freq_2_30,
        axis=1
    )

print("Frequency 2-30 created.")

Frequency 2-30 created.


In [150]:
# Block 124: Candidate volume for address-token frequency 2-30

freq_2_30_index = defaultdict(list)

for df in [s2, s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_2_30"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                freq_2_30_index[key].append(
                    row["entity_id"]
                )


freq_2_30_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_2_30"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                freq_2_30_index.get(key, [])
            )

    freq_2_30_candidates.append(candidate_ids)


freq_2_30_counts = np.array(
    [len(x) for x in freq_2_30_candidates]
)

print("=== Candidate Volume: Frequency 2-30 ===")
print("Index keys:", len(freq_2_30_index))
print("Average candidates/S1:", freq_2_30_counts.mean())
print("Max candidates/S1:", freq_2_30_counts.max())
print("Min candidates/S1:", freq_2_30_counts.min())
print("25%:", np.percentile(freq_2_30_counts, 25))
print("Median:", np.percentile(freq_2_30_counts, 50))
print("75%:", np.percentile(freq_2_30_counts, 75))
print(
    "S1 with >=1 candidate:",
    np.sum(freq_2_30_counts > 0)
)

total_possible = len(s2) + len(s3)

reduction_ratio_2_30 = 1 - (
    freq_2_30_counts.sum()
    / (len(s1) * total_possible)
)

print("Reduction ratio:", reduction_ratio_2_30)


=== Candidate Volume: Frequency 2-30 ===
Index keys: 25248
Average candidates/S1: 49.287
Max candidates/S1: 549
Min candidates/S1: 0
25%: 19.0
Median: 37.0
75%: 62.0
S1 with >=1 candidate: 9856
Reduction ratio: 0.99753565


In [151]:
# Block 125: Exact recall for
# Name-3 OR Address-prefix OR address-token frequency 2-30

true_freq_2_30_index = defaultdict(list)

for df in [true_s2, true_s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_2_30"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                true_freq_2_30_index[key].append(
                    row["entity_id"]
                )


true_freq_2_30_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_2_30"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                true_freq_2_30_index.get(key, [])
            )

    true_freq_2_30_candidates.append(candidate_ids)


recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    s1_position = s1.index[
        s1["entity_id"] == row["source1_entity_id"]
    ][0]

    captured = true_ids.intersection(
        true_freq_2_30_candidates[s1_position]
    )

    recall_values.append(
        len(captured) / len(true_ids)
    )

    if len(captured) == len(true_ids):
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1


print("=== Exact Recall: Frequency 2-30 ===")
print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print(
    "All true matches captured:",
    all_matches / evaluated
)
print(
    "At least one true match captured:",
    at_least_one / evaluated
)


=== Exact Recall: Frequency 2-30 ===
Evaluated S1: 9440
Mean recall: 0.9663342833602369
All true matches captured: 0.9029661016949152
At least one true match captured: 0.996822033898305


In [152]:
# Block 126: Address-token frequency 1-30

MIN_TOKEN_FREQ_1_30 = 1
MAX_TOKEN_FREQ_1_30 = 30

def address_tokens_freq_1_30(row):
    tokens = row["block_address_tokens_filtered"]
    return [
        token
        for token in tokens
        if MIN_TOKEN_FREQ_1_30 <= address_token_frequency[token] <= MAX_TOKEN_FREQ_1_30
    ]

for df in [s1, s2, s3, true_s2, true_s3]:
    df["block_address_tokens_freq_1_30"] = df.apply(
        address_tokens_freq_1_30,
        axis=1
    )

print("Frequency 1-30 created.")

Frequency 1-30 created.


In [153]:
# Block 127: Candidate volume for address-token frequency 1-30

freq_1_30_index = defaultdict(list)

for df in [s2, s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_1_30"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                freq_1_30_index[key].append(
                    row["entity_id"]
                )


freq_1_30_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_1_30"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                freq_1_30_index.get(key, [])
            )

    freq_1_30_candidates.append(candidate_ids)


freq_1_30_counts = np.array(
    [len(x) for x in freq_1_30_candidates]
)

print("=== Candidate Volume: Frequency 1-30 ===")
print("Index keys:", len(freq_1_30_index))
print("Average candidates/S1:", freq_1_30_counts.mean())
print("Max candidates/S1:", freq_1_30_counts.max())
print("Min candidates/S1:", freq_1_30_counts.min())
print("25%:", np.percentile(freq_1_30_counts, 25))
print("Median:", np.percentile(freq_1_30_counts, 50))
print("75%:", np.percentile(freq_1_30_counts, 75))
print(
    "S1 with >=1 candidate:",
    np.sum(freq_1_30_counts > 0)
)

total_possible = len(s2) + len(s3)

reduction_ratio_1_30 = 1 - (
    freq_1_30_counts.sum()
    / (len(s1) * total_possible)
)

print("Reduction ratio:", reduction_ratio_1_30)


=== Candidate Volume: Frequency 1-30 ===
Index keys: 39718
Average candidates/S1: 49.702
Max candidates/S1: 550
Min candidates/S1: 0
25%: 20.0
Median: 37.0
75%: 62.0
S1 with >=1 candidate: 9918
Reduction ratio: 0.9975149


In [154]:
# Block 128: Exact recall for
# Name-3 OR Address-prefix OR address-token frequency 1-30

true_freq_1_30_index = defaultdict(list)

for df in [true_s2, true_s3]:
    for _, row in df.iterrows():

        keys = [
            row["block_name_prefix_3"],
            row["block_address_prefix"]
        ]

        keys.extend(
            row["block_address_tokens_freq_1_30"]
        )

        for key in keys:
            if pd.notna(key) and key is not None:
                true_freq_1_30_index[key].append(
                    row["entity_id"]
                )


true_freq_1_30_candidates = []

for _, row in s1.iterrows():

    candidate_ids = set()

    keys = [
        row["block_name_prefix_3"],
        row["block_address_prefix"]
    ]

    keys.extend(
        row["block_address_tokens_freq_1_30"]
    )

    for key in keys:
        if pd.notna(key) and key is not None:
            candidate_ids.update(
                true_freq_1_30_index.get(key, [])
            )

    true_freq_1_30_candidates.append(candidate_ids)


recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    s1_position = s1.index[
        s1["entity_id"] == row["source1_entity_id"]
    ][0]

    captured = true_ids.intersection(
        true_freq_1_30_candidates[s1_position]
    )

    recall_values.append(
        len(captured) / len(true_ids)
    )

    if len(captured) == len(true_ids):
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1


print("=== Exact Recall: Frequency 1-30 ===")
print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print(
    "All true matches captured:",
    all_matches / evaluated
)
print(
    "At least one true match captured:",
    at_least_one / evaluated
)


=== Exact Recall: Frequency 1-30 ===
Evaluated S1: 9440
Mean recall: 0.9731031409739037
All true matches captured: 0.9220338983050848
At least one true match captured: 0.9973516949152542


In [155]:
from pathlib import Path

blocking_code = r'''
"""
Memory-conscious blocking / candidate generation for Amazon ML
Business Entity Resolution.

Final strategy:
    1. Country + compact normalized name prefix (3 chars)
    2. Country + normalized address prefix (6 chars)
    3. Country + informative address tokens with frequency 1-30

The rules are OR-ed together and candidates are deduplicated.

Public API:
    build_blocking_indexes(s2, s3)
    generate_candidates(s1_chunk, indexes)
"""

from __future__ import annotations

import re
import unicodedata
from collections import Counter, defaultdict
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd


NAME_PREFIX_LEN = 3
ADDRESS_PREFIX_LEN = 6

MIN_TOKEN_FREQ = 1
MAX_TOKEN_FREQ = 30

COMMON_ADDRESS_TOKENS = {
    "street", "st", "road", "rd", "avenue", "ave",
    "drive", "dr", "lane", "ln", "boulevard", "blvd",
    "highway", "hwy", "way", "place", "pl", "court", "ct",
    "circle", "cir", "parkway", "pkwy", "building", "block",
    "unit", "floor", "flat", "apartment", "apt", "house",
    "no", "number",
}


def normalize_text(value) -> str:
    if pd.isna(value):
        return ""

    value = str(value).lower()
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


def _compact_name(name: str) -> str:
    return name.replace(" ", "")


def _name_prefix_key(
    name: str,
    country: str,
    n: int = NAME_PREFIX_LEN,
) -> Optional[str]:

    if not name:
        return None

    compact = _compact_name(name)

    if len(compact) < n:
        return None

    return f"{country}|{compact[:n]}"


def _address_prefix_key(
    address: str,
    country: str,
    n: int = ADDRESS_PREFIX_LEN,
) -> Optional[str]:

    if not address:
        return None

    return f"{country}|{address[:n]}"


def _informative_address_tokens(
    address: str,
    country: str,
) -> List[str]:

    if not address:
        return []

    tokens = [
        token
        for token in address.split()
        if (
            len(token) >= 3
            and token not in COMMON_ADDRESS_TOKENS
            and not token.isdigit()
        )
    ]

    tokens = list(dict.fromkeys(tokens))

    return [
        f"{country}|{token}"
        for token in tokens
    ]


def _iter_rows(df: pd.DataFrame):

    required = {
        "entity_id",
        "business_name",
        "business_address",
        "country",
    }

    missing = required.difference(df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    for row in df.itertuples(index=False):
        yield (
            getattr(row, "entity_id"),
            getattr(row, "business_name"),
            getattr(row, "business_address"),
            getattr(row, "country"),
        )


def build_blocking_indexes(
    s2: pd.DataFrame,
    s3: pd.DataFrame,
    min_token_freq: int = MIN_TOKEN_FREQ,
    max_token_freq: int = MAX_TOKEN_FREQ,
) -> Dict:

    if min_token_freq < 1:
        raise ValueError("min_token_freq must be >= 1")

    if max_token_freq < min_token_freq:
        raise ValueError(
            "max_token_freq must be >= min_token_freq"
        )

    entity_ids = []
    rows = []
    address_token_frequency = Counter()

    # First pass: normalize and calculate token frequencies.
    for df in (s2, s3):

        for entity_id, raw_name, raw_address, raw_country in _iter_rows(df):

            name = normalize_text(raw_name)
            address = normalize_text(raw_address)
            country = normalize_text(raw_country)

            entity_ids.append(str(entity_id))

            rows.append(
                (
                    name,
                    address,
                    country,
                )
            )

            token_keys = _informative_address_tokens(
                address,
                country,
            )

            address_token_frequency.update(set(token_keys))

    allowed_address_tokens = {
        token
        for token, frequency in address_token_frequency.items()
        if min_token_freq <= frequency <= max_token_freq
    }

    # Second pass: build posting lists.
    key_to_indices = defaultdict(list)

    for row_idx, (name, address, country) in enumerate(rows):

        name_key = _name_prefix_key(
            name,
            country,
        )

        if name_key is not None:
            key_to_indices[name_key].append(row_idx)

        address_key = _address_prefix_key(
            address,
            country,
        )

        if address_key is not None:
            key_to_indices[address_key].append(row_idx)

        token_keys = _informative_address_tokens(
            address,
            country,
        )

        for token_key in token_keys:

            if token_key in allowed_address_tokens:
                key_to_indices[token_key].append(row_idx)

    return {
        "entity_ids": np.asarray(entity_ids, dtype=object),
        "key_to_indices": dict(key_to_indices),
        "address_token_frequency": dict(address_token_frequency),
        "allowed_address_tokens": allowed_address_tokens,
        "min_token_freq": min_token_freq,
        "max_token_freq": max_token_freq,
        "name_prefix_len": NAME_PREFIX_LEN,
        "address_prefix_len": ADDRESS_PREFIX_LEN,
    }


def generate_candidates(
    s1_chunk: pd.DataFrame,
    indexes: Dict,
) -> pd.DataFrame:

    required = {
        "entity_id",
        "business_name",
        "business_address",
        "country",
    }

    missing = required.difference(s1_chunk.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    entity_ids = indexes["entity_ids"]
    key_to_indices = indexes["key_to_indices"]

    output = []

    for row in s1_chunk.itertuples(index=False):

        source1_id = getattr(row, "entity_id")

        name = normalize_text(
            getattr(row, "business_name")
        )

        address = normalize_text(
            getattr(row, "business_address")
        )

        country = normalize_text(
            getattr(row, "country")
        )

        candidate_positions = set()

        # Name prefix
        name_key = _name_prefix_key(
            name,
            country,
        )

        if name_key is not None:
            positions = key_to_indices.get(name_key)

            if positions:
                candidate_positions.update(positions)

        # Address prefix
        address_key = _address_prefix_key(
            address,
            country,
        )

        if address_key is not None:
            positions = key_to_indices.get(address_key)

            if positions:
                candidate_positions.update(positions)

        # Frequency-filtered address tokens
        for token_key in _informative_address_tokens(
            address,
            country,
        ):

            if token_key not in indexes["allowed_address_tokens"]:
                continue

            positions = key_to_indices.get(token_key)

            if positions:
                candidate_positions.update(positions)

        candidate_ids = sorted(
            {
                str(entity_ids[position])
                for position in candidate_positions
            }
        )

        output.append(
            {
                "source1_entity_id": str(source1_id),
                "candidate_entity_ids": ",".join(candidate_ids),
            }
        )

    return pd.DataFrame(
        output,
        columns=[
            "source1_entity_id",
            "candidate_entity_ids",
        ],
    )


def generate_candidates_in_chunks(
    s1,
    indexes: Dict,
    chunk_size: int = 100_000,
) -> Iterable[pd.DataFrame]:

    if isinstance(s1, pd.DataFrame):

        for start in range(
            0,
            len(s1),
            chunk_size,
        ):

            chunk = s1.iloc[
                start:start + chunk_size
            ]

            yield generate_candidates(
                chunk,
                indexes,
            )

    else:

        for chunk in s1:

            yield generate_candidates(
                chunk,
                indexes,
            )
'''

blocking_path = Path(
    r"C:\Users\MY PC\Desktop\amazon-ml-entity-resolution\src\blocking.py"
)

blocking_path.write_text(
    blocking_code,
    encoding="utf-8"
)

print("Created:", blocking_path)
print("Lines:", len(blocking_code.splitlines()))

Created: C:\Users\MY PC\Desktop\amazon-ml-entity-resolution\src\blocking.py
Lines: 369


In [156]:
# Block 130: Import final blocking module

import sys
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\MY PC\Desktop\amazon-ml-entity-resolution"
)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.blocking import (
    normalize_text,
    build_blocking_indexes,
    generate_candidates,
)

print("blocking.py imported successfully.")
print("Name prefix length:", 3)
print("Address prefix length:", 6)
print("Address token frequency:", "1-30")

blocking.py imported successfully.
Name prefix length: 3
Address prefix length: 6
Address token frequency: 1-30


In [157]:
# Block 131: Build blocking indexes using the actual src/blocking.py

final_indexes = build_blocking_indexes(
    s2,
    s3,
    min_token_freq=1,
    max_token_freq=30,
)

print("=== Final Blocking Index ===")
print(
    "S2 rows:",
    len(s2)
)
print(
    "S3 rows:",
    len(s3)
)
print(
    "Total indexed entities:",
    len(final_indexes["entity_ids"])
)
print(
    "Blocking keys:",
    len(final_indexes["key_to_indices"])
)
print(
    "Allowed address tokens:",
    len(final_indexes["allowed_address_tokens"])
)
print(
    "Token frequency range:",
    final_indexes["min_token_freq"],
    "-",
    final_indexes["max_token_freq"]
)

=== Final Blocking Index ===
S2 rows: 10000
S3 rows: 10000
Total indexed entities: 20000
Blocking keys: 39718
Allowed address tokens: 21091
Token frequency range: 1 - 30


In [158]:
# Block 132: Generate candidates using the actual blocking module

final_candidates_df = generate_candidates(
    s1,
    final_indexes
)

print("=== Final Candidate Generation ===")
print("S1 rows:", len(s1))
print("Output rows:", len(final_candidates_df))

print("\nColumns:")
print(final_candidates_df.columns.tolist())

candidate_counts_final = (
    final_candidates_df["candidate_entity_ids"]
    .apply(
        lambda x: 0 if not x else len(x.split(","))
    )
)

print("\nCandidate statistics:")
print("Average candidates/S1:", candidate_counts_final.mean())
print("Max candidates/S1:", candidate_counts_final.max())
print("Min candidates/S1:", candidate_counts_final.min())
print("25%:", np.percentile(candidate_counts_final, 25))
print("Median:", np.percentile(candidate_counts_final, 50))
print("75%:", np.percentile(candidate_counts_final, 75))
print(
    "S1 with >=1 candidate:",
    np.sum(candidate_counts_final > 0)
)

total_possible = len(s2) + len(s3)

final_reduction_ratio = 1 - (
    candidate_counts_final.sum()
    / (len(s1) * total_possible)
)

print("Reduction ratio:", final_reduction_ratio)


=== Final Candidate Generation ===
S1 rows: 10000
Output rows: 10000

Columns:
['source1_entity_id', 'candidate_entity_ids']

Candidate statistics:
Average candidates/S1: 49.702
Max candidates/S1: 550
Min candidates/S1: 0
25%: 20.0
Median: 37.0
75%: 62.0
S1 with >=1 candidate: 9918
Reduction ratio: 0.9975149


In [159]:
# Block 133: Final recall validation using src/blocking.py

candidate_lookup = dict(
    zip(
        final_candidates_df["source1_entity_id"],
        final_candidates_df["candidate_entity_ids"]
    )
)

recall_values = []
all_matches = 0
at_least_one = 0
evaluated = 0

for _, row in gt_sample.iterrows():

    true_ids = row["true_match_ids"]

    if not true_ids:
        continue

    evaluated += 1

    candidate_string = candidate_lookup.get(
        str(row["source1_entity_id"]),
        ""
    )

    candidate_ids = set()

    if candidate_string:
        candidate_ids = set(
            candidate_string.split(",")
        )

    captured = true_ids.intersection(candidate_ids)

    recall_values.append(
        len(captured) / len(true_ids)
    )

    if len(captured) == len(true_ids):
        all_matches += 1

    if len(captured) > 0:
        at_least_one += 1


print("=== FINAL BLOCKING MODULE RECALL ===")
print("Evaluated S1:", evaluated)
print("Mean recall:", np.mean(recall_values))
print(
    "All true matches captured:",
    all_matches / evaluated
)
print(
    "At least one true match captured:",
    at_least_one / evaluated
)

=== FINAL BLOCKING MODULE RECALL ===
Evaluated S1: 9440
Mean recall: 0.0020095591202582723
All true matches captured: 0.0001059322033898305
At least one true match captured: 0.007097457627118644


In [160]:
# Block 134: Diagnose candidate ID vs ground-truth ID format

print("=== ID FORMAT DIAGNOSTIC ===")

# First S1 with a true match
example_row = gt_sample[gt_sample["true_match_ids"].apply(len) > 0].iloc[0]

s1_id = str(example_row["source1_entity_id"])
true_ids = example_row["true_match_ids"]

candidate_string = candidate_lookup.get(s1_id, "")
candidate_ids = (
    set(candidate_string.split(","))
    if candidate_string
    else set()
)

print("S1 ID:")
print(repr(s1_id))

print("\nTrue match IDs (first 5):")
print([repr(x) for x in list(true_ids)[:5]])

print("\nCandidate IDs (first 5):")
print([repr(x) for x in list(candidate_ids)[:5]])

print("\nTypes:")
print("true ID type:", type(next(iter(true_ids))) if true_ids else None)
print("candidate ID type:", type(next(iter(candidate_ids))) if candidate_ids else None)

print("\nIntersection:")
print(true_ids.intersection(candidate_ids))

print("\nCandidate count:", len(candidate_ids))
print("True match count:", len(true_ids))


=== ID FORMAT DIAGNOSTIC ===
S1 ID:
'S1-424976769'

True match IDs (first 5):
["'S3-296472077'", "'S3-443109567'", "'S2-18126063'"]

Candidate IDs (first 5):
["'S2-888124266'", "'S2-401293041'", "'S3-324883769'", "'S3-796387112'", "'S2-146349824'"]

Types:
true ID type: <class 'str'>
candidate ID type: <class 'str'>

Intersection:
set()

Candidate count: 83
True match count: 3


In [161]:
# Block 135: Verify src/blocking.py against the earlier notebook implementation

mismatches = 0
total_symmetric_difference = 0
max_symmetric_difference = 0

for i in range(len(s1)):

    # Candidates from src/blocking.py
    actual_string = final_candidates_df.iloc[i]["candidate_entity_ids"]

    actual = (
        set(actual_string.split(","))
        if actual_string
        else set()
    )

    # Candidates from our earlier notebook experiment
    expected = set(freq_1_30_candidates[i])

    difference = actual.symmetric_difference(expected)

    if difference:
        mismatches += 1
        total_symmetric_difference += len(difference)
        max_symmetric_difference = max(
            max_symmetric_difference,
            len(difference)
        )

print("=== BLOCKING IMPLEMENTATION VALIDATION ===")
print("S1 rows checked:", len(s1))
print("Rows with candidate-set mismatch:", mismatches)
print("Total differing candidate IDs:", total_symmetric_difference)
print("Maximum difference for one S1:", max_symmetric_difference)

print("\nImplementation matches notebook strategy:",
      mismatches == 0)

=== BLOCKING IMPLEMENTATION VALIDATION ===
S1 rows checked: 10000
Rows with candidate-set mismatch: 0
Total differing candidate IDs: 0
Maximum difference for one S1: 0

Implementation matches notebook strategy: True


In [162]:
# Block 136: Benchmark current blocking index memory usage

import time
import tracemalloc

print("=== CURRENT INDEX MEMORY BENCHMARK ===")

tracemalloc.start()
start_time = time.perf_counter()

benchmark_indexes = build_blocking_indexes(
    s2,
    s3,
    min_token_freq=1,
    max_token_freq=30
)

elapsed = time.perf_counter() - start_time

current_memory, peak_memory = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Build time: {elapsed:.3f} seconds")
print(f"Current Python memory: {current_memory / 1024**2:.2f} MB")
print(f"Peak Python memory: {peak_memory / 1024**2:.2f} MB")

print("\nIndex:")
print("Total entities:", len(benchmark_indexes["entity_ids"]))
print("Blocking keys:", len(benchmark_indexes["key_to_indices"]))
print(
    "Allowed address tokens:",
    len(benchmark_indexes["allowed_address_tokens"])
)

=== CURRENT INDEX MEMORY BENCHMARK ===
Build time: 3.589 seconds
Current Python memory: 11.89 MB
Peak Python memory: 19.02 MB

Index:
Total entities: 20000
Blocking keys: 39718
Allowed address tokens: 21091


In [163]:
# Block 137: Inspect the current blocking implementation

from pathlib import Path

blocking_path = Path("../src/blocking.py")

print(blocking_path.resolve())
print("=" * 80)

print(blocking_path.read_text(encoding="utf-8"))

C:\Users\MY PC\Desktop\amazon-ml-entity-resolution\src\blocking.py

"""
Memory-conscious blocking / candidate generation for Amazon ML
Business Entity Resolution.

Final strategy:
    1. Country + compact normalized name prefix (3 chars)
    2. Country + normalized address prefix (6 chars)
    3. Country + informative address tokens with frequency 1-30

The rules are OR-ed together and candidates are deduplicated.

Public API:
    build_blocking_indexes(s2, s3)
    generate_candidates(s1_chunk, indexes)
"""

from __future__ import annotations

import re
import unicodedata
from collections import Counter, defaultdict
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd


NAME_PREFIX_LEN = 3
ADDRESS_PREFIX_LEN = 6

MIN_TOKEN_FREQ = 1
MAX_TOKEN_FREQ = 30

COMMON_ADDRESS_TOKENS = {
    "street", "st", "road", "rd", "avenue", "ave",
    "drive", "dr", "lane", "ln", "boulevard", "blvd",
    "highway", "hwy", "way", "place", "pl", "court", "ct",
    "circl

In [164]:
# Block 138: Memory-optimized blocking.py
#
# Same blocking rules and public API.
# Removes the temporary normalized `rows` list.

from pathlib import Path

blocking_path = Path("../src/blocking.py")

optimized_code = r'''
"""
Memory-conscious blocking / candidate generation for Amazon ML
Business Entity Resolution.

Final strategy:
    1. Country + compact normalized name prefix (3 chars)
    2. Country + normalized address prefix (6 chars)
    3. Country + informative address tokens with frequency 1-30

The rules are OR-ed together and candidates are deduplicated.

Public API:
    build_blocking_indexes(s2, s3)
    generate_candidates(s1_chunk, indexes)
"""

from __future__ import annotations

import re
import unicodedata
from collections import Counter, defaultdict
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd


NAME_PREFIX_LEN = 3
ADDRESS_PREFIX_LEN = 6

MIN_TOKEN_FREQ = 1
MAX_TOKEN_FREQ = 30

COMMON_ADDRESS_TOKENS = {
    "street", "st", "road", "rd", "avenue", "ave",
    "drive", "dr", "lane", "ln", "boulevard", "blvd",
    "highway", "hwy", "way", "place", "pl", "court", "ct",
    "circle", "cir", "parkway", "pkwy", "building", "block",
    "unit", "floor", "flat", "apartment", "apt", "house",
    "no", "number",
}


def normalize_text(value) -> str:
    if pd.isna(value):
        return ""

    value = str(value).lower()
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


def _compact_name(name: str) -> str:
    return name.replace(" ", "")


def _name_prefix_key(
    name: str,
    country: str,
    n: int = NAME_PREFIX_LEN,
) -> Optional[str]:

    if not name:
        return None

    compact = _compact_name(name)

    if len(compact) < n:
        return None

    return f"{country}|{compact[:n]}"


def _address_prefix_key(
    address: str,
    country: str,
    n: int = ADDRESS_PREFIX_LEN,
) -> Optional[str]:

    if not address:
        return None

    return f"{country}|{address[:n]}"


def _informative_address_tokens(
    address: str,
    country: str,
) -> List[str]:

    if not address:
        return []

    tokens = [
        token
        for token in address.split()
        if (
            len(token) >= 3
            and token not in COMMON_ADDRESS_TOKENS
            and not token.isdigit()
        )
    ]

    tokens = list(dict.fromkeys(tokens))

    return [
        f"{country}|{token}"
        for token in tokens
    ]


def _iter_rows(df: pd.DataFrame):

    required = {
        "entity_id",
        "business_name",
        "business_address",
        "country",
    }

    missing = required.difference(df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    for row in df.itertuples(index=False):
        yield (
            getattr(row, "entity_id"),
            getattr(row, "business_name"),
            getattr(row, "business_address"),
            getattr(row, "country"),
        )


def build_blocking_indexes(
    s2: pd.DataFrame,
    s3: pd.DataFrame,
    min_token_freq: int = MIN_TOKEN_FREQ,
    max_token_freq: int = MAX_TOKEN_FREQ,
) -> Dict:

    if min_token_freq < 1:
        raise ValueError("min_token_freq must be >= 1")

    if max_token_freq < min_token_freq:
        raise ValueError(
            "max_token_freq must be >= min_token_freq"
        )

    entity_ids = []
    address_token_frequency = Counter()

    # First pass:
    #   - collect entity IDs
    #   - calculate address-token frequencies
    #
    # We intentionally do NOT store normalized rows here.
    # The source DataFrames are already available for the second pass.

    for df in (s2, s3):

        for entity_id, raw_name, raw_address, raw_country in _iter_rows(df):

            address = normalize_text(raw_address)
            country = normalize_text(raw_country)

            entity_ids.append(str(entity_id))

            token_keys = _informative_address_tokens(
                address,
                country,
            )

            address_token_frequency.update(set(token_keys))

    allowed_address_tokens = {
        token
        for token, frequency in address_token_frequency.items()
        if min_token_freq <= frequency <= max_token_freq
    }

    # Second pass:
    # Build posting lists directly from the source DataFrames.

    key_to_indices = defaultdict(list)
    row_idx = 0

    for df in (s2, s3):

        for entity_id, raw_name, raw_address, raw_country in _iter_rows(df):

            name = normalize_text(raw_name)
            address = normalize_text(raw_address)
            country = normalize_text(raw_country)

            # Name prefix
            name_key = _name_prefix_key(
                name,
                country,
            )

            if name_key is not None:
                key_to_indices[name_key].append(row_idx)

            # Address prefix
            address_key = _address_prefix_key(
                address,
                country,
            )

            if address_key is not None:
                key_to_indices[address_key].append(row_idx)

            # Frequency-filtered address tokens
            token_keys = _informative_address_tokens(
                address,
                country,
            )

            for token_key in token_keys:

                if token_key in allowed_address_tokens:
                    key_to_indices[token_key].append(row_idx)

            row_idx += 1

    return {
        "entity_ids": np.asarray(entity_ids, dtype=object),
        "key_to_indices": dict(key_to_indices),
        "address_token_frequency": dict(address_token_frequency),
        "allowed_address_tokens": allowed_address_tokens,
        "min_token_freq": min_token_freq,
        "max_token_freq": max_token_freq,
        "name_prefix_len": NAME_PREFIX_LEN,
        "address_prefix_len": ADDRESS_PREFIX_LEN,
    }


def generate_candidates(
    s1_chunk: pd.DataFrame,
    indexes: Dict,
) -> pd.DataFrame:

    required = {
        "entity_id",
        "business_name",
        "business_address",
        "country",
    }

    missing = required.difference(s1_chunk.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    entity_ids = indexes["entity_ids"]
    key_to_indices = indexes["key_to_indices"]

    output = []

    for row in s1_chunk.itertuples(index=False):

        source1_id = getattr(row, "entity_id")

        name = normalize_text(
            getattr(row, "business_name")
        )

        address = normalize_text(
            getattr(row, "business_address")
        )

        country = normalize_text(
            getattr(row, "country")
        )

        candidate_positions = set()

        # Name prefix
        name_key = _name_prefix_key(
            name,
            country,
        )

        if name_key is not None:
            positions = key_to_indices.get(name_key)

            if positions:
                candidate_positions.update(positions)

        # Address prefix
        address_key = _address_prefix_key(
            address,
            country,
        )

        if address_key is not None:
            positions = key_to_indices.get(address_key)

            if positions:
                candidate_positions.update(positions)

        # Frequency-filtered address tokens
        for token_key in _informative_address_tokens(
            address,
            country,
        ):

            if token_key not in indexes["allowed_address_tokens"]:
                continue

            positions = key_to_indices.get(token_key)

            if positions:
                candidate_positions.update(positions)

        candidate_ids = sorted(
            {
                str(entity_ids[position])
                for position in candidate_positions
            }
        )

        output.append(
            {
                "source1_entity_id": str(source1_id),
                "candidate_entity_ids": ",".join(candidate_ids),
            }
        )

    return pd.DataFrame(
        output,
        columns=[
            "source1_entity_id",
            "candidate_entity_ids",
        ],
    )


def generate_candidates_in_chunks(
    s1,
    indexes: Dict,
    chunk_size: int = 100_000,
) -> Iterable[pd.DataFrame]:

    if isinstance(s1, pd.DataFrame):

        for start in range(
            0,
            len(s1),
            chunk_size,
        ):

            chunk = s1.iloc[
                start:start + chunk_size
            ]

            yield generate_candidates(
                chunk,
                indexes,
            )

    else:

        for chunk in s1:

            yield generate_candidates(
                chunk,
                indexes,
            )
'''

blocking_path.write_text(
    optimized_code.strip() + "\n",
    encoding="utf-8",
)

print("Updated:", blocking_path.resolve())
print("Memory optimization applied.")

Updated: C:\Users\MY PC\Desktop\amazon-ml-entity-resolution\src\blocking.py
Memory optimization applied.


In [165]:
# Block 139: Reload optimized module and validate candidate-set equivalence

import importlib
import sys

# Reload the module from the updated src/blocking.py
import src.blocking as blocking
blocking = importlib.reload(blocking)

print("=== OPTIMIZED MODULE LOADED ===")
print("Module:", blocking.__file__)

# Rebuild the index using the optimized implementation
optimized_indexes = blocking.build_blocking_indexes(
    s2,
    s3,
    min_token_freq=1,
    max_token_freq=30
)

# Generate candidates using the optimized implementation
optimized_candidates_df = blocking.generate_candidates(
    s1,
    optimized_indexes
)

# Compare against the previously validated candidate sets
mismatches = 0
total_symmetric_difference = 0
max_symmetric_difference = 0

for i in range(len(s1)):

    actual_string = optimized_candidates_df.iloc[i][
        "candidate_entity_ids"
    ]

    actual = (
        set(actual_string.split(","))
        if actual_string
        else set()
    )

    expected = set(freq_1_30_candidates[i])

    difference = actual.symmetric_difference(expected)

    if difference:
        mismatches += 1
        total_symmetric_difference += len(difference)
        max_symmetric_difference = max(
            max_symmetric_difference,
            len(difference)
        )

print("\n=== OPTIMIZED IMPLEMENTATION VALIDATION ===")
print("S1 rows checked:", len(s1))
print("Rows with candidate-set mismatch:", mismatches)
print("Total differing candidate IDs:", total_symmetric_difference)
print("Maximum difference for one S1:", max_symmetric_difference)

print(
    "\nImplementation matches notebook strategy:",
    mismatches == 0
)

# Also verify candidate counts
optimized_counts = (
    optimized_candidates_df["candidate_entity_ids"]
    .apply(lambda x: 0 if not x else len(x.split(",")))
)

print("\n=== CANDIDATE STATISTICS ===")
print("Average:", optimized_counts.mean())
print("Max:", optimized_counts.max())
print("Min:", optimized_counts.min())
print("Median:", optimized_counts.median())
print("S1 with >=1:", (optimized_counts > 0).sum())

=== OPTIMIZED MODULE LOADED ===
Module: C:\Users\MY PC\Desktop\amazon-ml-entity-resolution\src\blocking.py

=== OPTIMIZED IMPLEMENTATION VALIDATION ===
S1 rows checked: 10000
Rows with candidate-set mismatch: 0
Total differing candidate IDs: 0
Maximum difference for one S1: 0

Implementation matches notebook strategy: True

=== CANDIDATE STATISTICS ===
Average: 49.702
Max: 550
Min: 0
Median: 37.0
S1 with >=1: 9918


In [166]:
# Block 140: Benchmark optimized implementation

import time
import tracemalloc

print("=== OPTIMIZED INDEX MEMORY BENCHMARK ===")

tracemalloc.start()
start_time = time.perf_counter()

benchmark_optimized_indexes = blocking.build_blocking_indexes(
    s2,
    s3,
    min_token_freq=1,
    max_token_freq=30
)

elapsed = time.perf_counter() - start_time

current_memory, peak_memory = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Build time: {elapsed:.3f} seconds")
print(f"Current Python memory: {current_memory / 1024**2:.2f} MB")
print(f"Peak Python memory: {peak_memory / 1024**2:.2f} MB")

print("\nIndex:")
print(
    "Total entities:",
    len(benchmark_optimized_indexes["entity_ids"])
)
print(
    "Blocking keys:",
    len(benchmark_optimized_indexes["key_to_indices"])
)
print(
    "Allowed address tokens:",
    len(
        benchmark_optimized_indexes[
            "allowed_address_tokens"
        ]
    )
)

=== OPTIMIZED INDEX MEMORY BENCHMARK ===
Build time: 7.669 seconds
Current Python memory: 11.77 MB
Peak Python memory: 13.75 MB

Index:
Total entities: 20000
Blocking keys: 39718
Allowed address tokens: 21091


In [167]:
# Block 141: Validate chunked candidate generation

print("=== CHUNKED CANDIDATE GENERATION TEST ===")

chunk_size = 2000

chunk_outputs = list(
    blocking.generate_candidates_in_chunks(
        s1,
        optimized_indexes,
        chunk_size=chunk_size
    )
)

chunked_candidates_df = pd.concat(
    chunk_outputs,
    ignore_index=True
)

print("Number of chunks:", len(chunk_outputs))
print("Rows produced:", len(chunked_candidates_df))

# Compare chunked output with normal output
normal_output = optimized_candidates_df.reset_index(drop=True)
chunked_output = chunked_candidates_df.reset_index(drop=True)

same_output = normal_output.equals(chunked_output)

print("Chunked output identical to normal output:", same_output)

print("\nChunk sizes:")
print([len(chunk) for chunk in chunk_outputs])

=== CHUNKED CANDIDATE GENERATION TEST ===
Number of chunks: 5
Rows produced: 10000
Chunked output identical to normal output: True

Chunk sizes:
[2000, 2000, 2000, 2000, 2000]


In [168]:
# Block 142: Final Git status and diff check

import subprocess

repo_dir = Path(r"C:\Users\MY PC\Desktop\amazon-ml-entity-resolution")

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo_dir,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

print("=== GIT STATUS ===")
run_git("status", "--short")

print("\n=== CURRENT BRANCH ===")
run_git("branch", "--show-current")

print("\n=== DIFF STAT ===")
run_git("diff", "--stat")

print("\n=== DIFF ===")
run_git("diff", "--", "src/blocking.py")

=== GIT STATUS ===
?? anaconda_projects/
?? notebooks/blocking_experiments.ipynb
?? src/blocking.py


=== CURRENT BRANCH ===
feature-blocking


=== DIFF STAT ===


=== DIFF ===



0

In [169]:
# Block 143: Inspect untracked files before staging

print("=== UNTRACKED FILES ===")

result = subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

print(result.stdout)

print("\n=== ANACONDA_PROJECTS CONTENTS ===")

anaconda_dir = repo_dir / "anaconda_projects"

if anaconda_dir.exists():
    for path in list(anaconda_dir.rglob("*"))[:50]:
        relative = path.relative_to(repo_dir)
        print(relative)
else:
    print("anaconda_projects/ does not exist")

=== UNTRACKED FILES ===
?? anaconda_projects/db/project_filebrowser.db
?? notebooks/blocking_experiments.ipynb
?? src/blocking.py


=== ANACONDA_PROJECTS CONTENTS ===
anaconda_projects\db
anaconda_projects\db\project_filebrowser.db


In [170]:
# Block 144: Final pre-stage file check

blocking_file = repo_dir / "src" / "blocking.py"
notebook_file = repo_dir / "notebooks" / "blocking_experiments.ipynb"

print("=== FILE SIZES ===")
print(
    "src/blocking.py:",
    round(blocking_file.stat().st_size / 1024, 2),
    "KB"
)
print(
    "notebooks/blocking_experiments.ipynb:",
    round(notebook_file.stat().st_size / 1024, 2),
    "KB"
)

print("\n=== NOTEBOOK CHECKPOINT ===")

import json

with open(notebook_file, "r", encoding="utf-8") as f:
    notebook_json = json.load(f)

print("Notebook cells:", len(notebook_json.get("cells", [])))
print("Notebook format:", notebook_json.get("nbformat"))

print("\n=== GITIGNORE CHECK ===")

gitignore_file = repo_dir / ".gitignore"

if gitignore_file.exists():
    print(gitignore_file.read_text(encoding="utf-8"))
else:
    print(".gitignore does not exist")

=== FILE SIZES ===
src/blocking.py: 8.91 KB
notebooks/blocking_experiments.ipynb: 354.46 KB

=== NOTEBOOK CHECKPOINT ===
Notebook cells: 142
Notebook format: 4

=== GITIGNORE CHECK ===
# Python
__pycache__/
*.py[cod]
*.so

# Jupyter
.ipynb_checkpoints/

# Virtual environments
.venv/
venv/
env/

# Local datasets - NEVER commit challenge data
amazonml_data/
*.tsv
*.csv

# Generated outputs
output/*
!output/.gitkeep

# OS files
.DS_Store

# Secrets
.env
*.pem



In [171]:
# Block 145: Stage only the blocking implementation and notebook

run_git("add", "src/blocking.py", "notebooks/blocking_experiments.ipynb")

print("=== STAGED FILES ===")
run_git("status", "--short")

print("\n=== STAGED DIFF STAT ===")
run_git("diff", "--cached", "--stat")



=== STAGED FILES ===
A  notebooks/blocking_experiments.ipynb
A  src/blocking.py
?? anaconda_projects/


=== STAGED DIFF STAT ===
 notebooks/blocking_experiments.ipynb | 11221 +++++++++++++++++++++++++++++++++
 src/blocking.py                      |   378 ++
 2 files changed, 11599 insertions(+)



0

In [172]:
# Block 146: Final staged-content safety check

print("=== STAGED FILES ===")
run_git("diff", "--cached", "--name-status")

print("\n=== STAGED TSV/CSV CHECK ===")

result = subprocess.run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

staged_files = [
    line.strip()
    for line in result.stdout.splitlines()
    if line.strip()
]

data_files = [
    path for path in staged_files
    if path.lower().endswith((".tsv", ".csv"))
]

print("Staged data files:", data_files)

print("\n=== BLOCKING.PY OPTIMIZATION CHECK ===")

result = subprocess.run(
    [
        "git",
        "show",
        ":src/blocking.py",
    ],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

blocking_staged = result.stdout

print(
    "Temporary normalized rows list present:",
    "rows = []" in blocking_staged
)

print(
    "Second-pass source iteration present:",
    "for df in (s2, s3):" in blocking_staged
)

=== STAGED FILES ===
A	notebooks/blocking_experiments.ipynb
A	src/blocking.py


=== STAGED TSV/CSV CHECK ===
Staged data files: []

=== BLOCKING.PY OPTIMIZATION CHECK ===
Temporary normalized rows list present: False
Second-pass source iteration present: True


In [173]:
# Block 147: Commit the completed blocking implementation

commit_result = subprocess.run(
    [
        "git",
        "commit",
        "-m",
        "Implement blocking candidate generation",
    ],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

print(commit_result.stdout)

if commit_result.stderr:
    print(commit_result.stderr)

print("\nCommit return code:", commit_result.returncode)

[feature-blocking b5b0a7d] Implement blocking candidate generation
 2 files changed, 11599 insertions(+)
 create mode 100644 notebooks/blocking_experiments.ipynb
 create mode 100644 src/blocking.py


Commit return code: 0


In [174]:
# Block 148: Verify commit and working tree

print("=== CURRENT BRANCH ===")
run_git("branch", "--show-current")

print("\n=== LATEST COMMIT ===")
run_git("log", "-1", "--oneline")

print("\n=== WORKING TREE STATUS ===")
run_git("status", "--short")

=== CURRENT BRANCH ===
feature-blocking


=== LATEST COMMIT ===
b5b0a7d Implement blocking candidate generation


=== WORKING TREE STATUS ===
 M notebooks/blocking_experiments.ipynb
?? anaconda_projects/



0

In [175]:
# Block 149: Inspect post-commit notebook changes

print("=== NOTEBOOK CHANGE SUMMARY ===")
run_git("diff", "--stat", "--", "notebooks/blocking_experiments.ipynb")

print("\n=== NOTEBOOK DIFF STATUS ===")
diff_result = subprocess.run(
    ["git", "diff", "--numstat", "--", "notebooks/blocking_experiments.ipynb"],
    cwd=repo_dir,
    capture_output=True,
    text=True
)
print(diff_result.stdout)

print("\n=== ANACONDA PROJECTS STATUS ===")
run_git("status", "--short", "--", "anaconda_projects/")

=== NOTEBOOK CHANGE SUMMARY ===
 notebooks/blocking_experiments.ipynb | 217 ++++++++++++++++++++++++++++++++++-
 1 file changed, 215 insertions(+), 2 deletions(-)



=== NOTEBOOK DIFF STATUS ===
215	2	notebooks/blocking_experiments.ipynb


=== ANACONDA PROJECTS STATUS ===
?? anaconda_projects/



0